# C3_genetic_algorithm_multiple_testing: Genetic algorithms and multiple-testing correction

Public appendix notebook from the Bachelor's Thesis *Evaluación robusta de estrategias de inversión: backtesting, sobreajuste y validación estadística*.

This notebook is included for transparency/reproducibility. It was originally developed in Google Colab; paths may need to be adapted for local execution.


In [ ]:
# -*- coding: utf-8 -*-
"""block3.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1xt-VNWo-DSb0h6T1TJ3rEgkcLhuEoVPE

Instalación
"""

!pip -q install pandas numpy yfinance matplotlib lxml html5lib beautifulsoup4 requests


In [ ]:
# ============================================================
# CELDA 1 - MONTAR GOOGLE DRIVE Y CREAR CARPETAS


In [ ]:
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Carpeta base del TFG en Drive
TFG_BASE_DIR = "/content/drive/MyDrive/TFG"

# Crear carpetas necesarias
Path(f"{TFG_BASE_DIR}").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data/cache").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/results").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/logs").mkdir(parents=True, exist_ok=True)

print("Google Drive montado.")
print(f"Carpeta base del TFG: {TFG_BASE_DIR}")
print("Carpetas creadas correctamente.")

"""Código completo del Bloque 0"""


In [ ]:
# ============================================================
# CAP 0 - MOTOR REPRODUCIBLE DE DATOS, BACKTESTING Y MÉTRICAS
# Versión Colab con reconstrucción histórica del S&P 100


In [ ]:
# ============================================================

from __future__ import annotations

import json
import math
import time
import warnings
import re
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from typing import Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import requests


In [ ]:
# ============================================================
# 1. CONFIGURACIÓN GENERAL


In [ ]:
# ============================================================

@dataclass
class BacktestConfig:
    # Periodo de análisis.
    # Yahoo Finance usa end_date como fecha final EXCLUSIVA.
    start_date: str = "2021-01-01"
    end_date: str = "2026-01-01"

    # Datos
    data_source: str = "Yahoo Finance via yfinance"
    universe_name: str = "S&P 100"

    # Archivo manual si algún día tenemos una fuente histórica propia.
    membership_file: str = "data/sp100_membership.csv"

    # Archivo reconstruido automáticamente desde revisiones históricas de Wikipedia.
    reconstructed_membership_file: str = "data/sp100_membership_reconstructed_wikipedia.csv"
    snapshot_log_file: str = "data/sp100_wikipedia_snapshot_log.csv"

    # Reconstrucción histórica
    use_reconstructed_membership: bool = True
    force_rebuild_membership: bool = True
    membership_history_frequency: str = "MS"

    # Importante:
    # Si esto está en False, el código NO vuelve a una lista actual si falla la reconstrucción.
    # Así evitamos meter survivorship bias sin querer.
    allow_current_fallback: bool = False

    # Benchmark
    # SPY = proxy invertible del S&P 500 con precios ajustados.
    # ^GSPC = índice S&P 500 de precio, no total return.
    benchmark_ticker: str = "SPY"
    benchmark_name: str = "S&P 500 benchmark"

    # Backtest
    initial_capital: float = 1.0
    trading_days_per_year: int = 252

    # Costes
    transaction_cost: float = 0.001
    cost_sensitivity: Tuple[float, ...] = (0.0, 0.0005, 0.001, 0.0025)

    # Tasa libre de riesgo anual provisional.
    risk_free_rate_annual: float = 0.0225

    # Limpieza
    min_price_coverage: float = 0.80
    min_assets_required: int = 20

    # Estrategia
    # "static_buy_hold": compra al inicio el universo histórico inicial y no rebalancea.
    # "dynamic_equal_weight": usa el universo histórico correspondiente en cada rebalanceo.
    strategy_mode: str = "dynamic_equal_weight"
    rebalance_frequency: str = "M"

    # Reproducibilidad
    random_seed: int = 123
    output_dir: str = "results/block0_sp100_survivorship_corrected"


CONFIG = BacktestConfig()


In [ ]:
# ============================================================
# 2. UTILIDADES GENERALES


In [ ]:
# ============================================================

def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def now_utc_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def save_json(obj: dict, path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4, ensure_ascii=False, default=str)


def to_yfinance_ticker(ticker: str) -> str:
    """
    Yahoo Finance usa '-' en vez de '.' para clases de acciones.
    Ejemplo: BRK.B -> BRK-B.
    """
    return str(ticker).strip().replace(".", "-")


def from_yfinance_ticker(ticker: str) -> str:
    return str(ticker).strip().replace("-", ".")


In [ ]:
# ============================================================
# 3. UNIVERSO S&P 100 HISTÓRICO


In [ ]:
# ============================================================

WIKIPEDIA_API_URL = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_TITLE_SP100 = "S&P 100"


# Lista solo de emergencia.
# Por defecto NO se usa porque allow_current_fallback=False.
SP100_FALLBACK_TICKERS = [
    "AAPL", "ABBV", "ABT", "ACN", "ADBE", "AIG", "AMD", "AMGN", "AMT", "AMZN",
    "AVGO", "AXP", "BA", "BAC", "BK", "BKNG", "BLK", "BMY", "BRK.B", "C",
    "CAT", "CHTR", "CL", "CMCSA", "COF", "COP", "COST", "CRM", "CSCO", "CVS",
    "CVX", "DE", "DHR", "DIS", "DUK", "EMR", "EXC", "F", "FDX", "GD",
    "GE", "GILD", "GM", "GOOG", "GOOGL", "GS", "HD", "HON", "IBM", "INTC",
    "JNJ", "JPM", "KHC", "KO", "LIN", "LLY", "LMT", "LOW", "MA", "MCD",
    "MDLZ", "MDT", "MET", "META", "MMM", "MO", "MRK", "MS", "MSFT", "NEE",
    "NFLX", "NKE", "NVDA", "ORCL", "PEP", "PFE", "PG", "PM", "PYPL", "QCOM",
    "RTX", "SBUX", "SCHW", "SO", "SPG", "T", "TGT", "TMO", "TXN", "UNH",
    "UNP", "UPS", "USB", "V", "VZ", "WBA", "WFC", "WMT", "XOM"
]


def clean_ticker_symbol(x: str) -> Optional[str]:
    """
    Limpia símbolos extraídos de tablas HTML.
    """
    if pd.isna(x):
        return None

    s = str(x).strip()
    s = re.sub(r"\[.*?\]", "", s)
    s = s.replace("\xa0", " ")
    s = s.strip()

    if not s:
        return None

    bad_values = {"symbol", "ticker", "nan", "none", "company", "security"}
    if s.lower() in bad_values:
        return None

    s = re.sub(r"[^A-Za-z0-9\.\-]", "", s)

    if not s:
        return None

    return s.upper()


def wikipedia_api_get(
    params: dict,
    max_retries: int = 4,
    sleep_seconds: float = 0.8,
) -> dict:
    """
    Petición robusta a la API de Wikipedia.
    """
    headers = {
        "User-Agent": "TFG-Backtest-Research/1.0 academic Colab notebook"
    }

    last_error = None

    for attempt in range(max_retries):
        try:
            response = requests.get(
                WIKIPEDIA_API_URL,
                params=params,
                headers=headers,
                timeout=30,
            )

            response.raise_for_status()
            data = response.json()

            if "error" in data:
                raise RuntimeError(data["error"])

            return data

        except Exception as e:
            last_error = e
            time.sleep(sleep_seconds * (attempt + 1))

    raise RuntimeError(f"No se pudo consultar Wikipedia API. Último error: {last_error}")


def get_wikipedia_revision_at(date: pd.Timestamp) -> Tuple[int, str]:
    """
    Obtiene la revisión de la página S&P 100 existente en una fecha concreta.
    """
    date = pd.Timestamp(date)
    rvstart = date.strftime("%Y-%m-%dT23:59:59Z")

    params = {
        "action": "query",
        "format": "json",
        "formatversion": "2",
        "prop": "revisions",
        "titles": WIKIPEDIA_TITLE_SP100,
        "rvprop": "ids|timestamp",
        "rvlimit": "1",
        "rvdir": "older",
        "rvstart": rvstart,
    }

    data = wikipedia_api_get(params)
    pages = data.get("query", {}).get("pages", [])

    if not pages or "revisions" not in pages[0] or not pages[0]["revisions"]:
        raise RuntimeError(f"No se encontró revisión de Wikipedia para {date.date()}")

    rev = pages[0]["revisions"][0]

    return int(rev["revid"]), str(rev["timestamp"])


def get_wikipedia_revision_html(revid: int) -> str:
    """
    Convierte una revisión concreta a HTML mediante action=parse.
    """
    params = {
        "action": "parse",
        "format": "json",
        "oldid": str(revid),
        "prop": "text",
    }

    data = wikipedia_api_get(params)
    html = data.get("parse", {}).get("text", {}).get("*")

    if not html:
        raise RuntimeError(f"No se pudo extraer HTML para oldid={revid}")

    return html


def extract_sp100_tickers_from_html(html: str) -> List[str]:
    """
    Extrae los tickers de la tabla de constituyentes del S&P 100.

    Busca tablas con columna Symbol/Ticker.
    Acepta una tabla si encuentra aproximadamente entre 80 y 120 símbolos.
    """
    tables = pd.read_html(StringIO(html))
    candidates = []

    for table in tables:
        table = table.copy()

        if isinstance(table.columns, pd.MultiIndex):
            table.columns = [
                " ".join([str(x) for x in col if str(x) != "nan"]).strip()
                for col in table.columns
            ]
        else:
            table.columns = [str(c).strip() for c in table.columns]

        symbol_col = None

        for col in table.columns:
            col_lower = str(col).lower()
            if "symbol" in col_lower or "ticker" in col_lower:
                symbol_col = col
                break

        if symbol_col is None:
            continue

        tickers = []

        for value in table[symbol_col].tolist():
            ticker = clean_ticker_symbol(value)
            if ticker is not None:
                tickers.append(ticker)

        tickers = sorted(set(tickers))

        if 80 <= len(tickers) <= 120:
            candidates.append(tickers)

    if not candidates:
        raise RuntimeError("No se encontró una tabla válida de constituyentes del S&P 100.")

    candidates = sorted(candidates, key=lambda x: abs(len(x) - 100))

    return candidates[0]


def fetch_sp100_snapshot_from_wikipedia(date: pd.Timestamp) -> Tuple[List[str], dict]:
    """
    Descarga la composición del S&P 100 según la página de Wikipedia
    tal como estaba en la fecha indicada.
    """
    revid, revision_timestamp = get_wikipedia_revision_at(date)
    html = get_wikipedia_revision_html(revid)
    tickers = extract_sp100_tickers_from_html(html)

    metadata = {
        "snapshot_date": str(pd.Timestamp(date).date()),
        "wiki_revision_id": revid,
        "wiki_revision_timestamp": revision_timestamp,
        "n_tickers": len(tickers),
        "status": "ok",
    }

    return tickers, metadata


def make_snapshot_dates(start_date: str, end_date: str, freq: str) -> List[pd.Timestamp]:
    """
    Genera fechas de snapshot entre start_date y end_date.

    end_date se considera exclusiva, igual que en Yahoo Finance.
    """
    start = pd.Timestamp(start_date).normalize()
    end_exclusive = pd.Timestamp(end_date).normalize()
    end_inclusive = end_exclusive - pd.Timedelta(days=1)

    regular_dates = pd.date_range(start=start, end=end_inclusive, freq=freq)

    dates = sorted(set([start] + list(regular_dates)))
    dates = [pd.Timestamp(d).normalize() for d in dates if start <= d <= end_inclusive]

    return dates


def snapshots_to_membership(snapshot_rows: List[dict]) -> pd.DataFrame:
    """
    Convierte snapshots mensuales en intervalos de pertenencia.
    """
    snapshots = pd.DataFrame(snapshot_rows)

    if snapshots.empty:
        raise RuntimeError("No hay snapshots para construir membership.")

    snapshots["snapshot_date"] = pd.to_datetime(snapshots["snapshot_date"])
    snapshots = snapshots.sort_values("snapshot_date")

    interval_rows = []
    unique_dates = sorted(snapshots["snapshot_date"].unique())

    for i, date in enumerate(unique_dates):
        current = snapshots[snapshots["snapshot_date"] == date]
        tickers = sorted(current["ticker"].dropna().unique().tolist())

        if i < len(unique_dates) - 1:
            interval_end = pd.Timestamp(unique_dates[i + 1]) - pd.Timedelta(days=1)
        else:
            interval_end = pd.NaT

        for ticker in tickers:
            interval_rows.append({
                "ticker": ticker,
                "yf_ticker": to_yfinance_ticker(ticker),
                "start": pd.Timestamp(date),
                "end": interval_end,
                "source": "wikipedia_historical_revision_snapshot",
            })

    membership = pd.DataFrame(interval_rows)

    if membership.empty:
        raise RuntimeError("Membership histórico vacío.")

    return membership


def reconstruct_sp100_membership_from_wikipedia(
    config: BacktestConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Reconstruye la composición histórica del S&P 100 usando revisiones históricas
    de Wikipedia.

    Esto evita usar una lista actual para todo el pasado.
    """
    snapshot_dates = make_snapshot_dates(
        config.start_date,
        config.end_date,
        config.membership_history_frequency,
    )

    print("Reconstruyendo S&P 100 histórico desde Wikipedia.")
    print(f"Snapshots: {len(snapshot_dates)} fechas ({config.membership_history_frequency})")

    snapshot_rows = []
    snapshot_log = []

    last_good_tickers = None

    for i, date in enumerate(snapshot_dates):
        print(f"  Snapshot {i + 1}/{len(snapshot_dates)}: {date.date()}")

        try:
            tickers, meta = fetch_sp100_snapshot_from_wikipedia(date)
            last_good_tickers = tickers

        except Exception as e:
            if last_good_tickers is None:
                raise RuntimeError(
                    f"Falló el primer snapshot ({date.date()}) y no hay composición anterior para arrastrar. "
                    f"Error: {e}"
                )

            tickers = last_good_tickers
            meta = {
                "snapshot_date": str(date.date()),
                "wiki_revision_id": None,
                "wiki_revision_timestamp": None,
                "n_tickers": len(tickers),
                "status": f"carried_forward_after_error: {type(e).__name__}: {e}",
            }

        for ticker in tickers:
            snapshot_rows.append({
                "snapshot_date": str(date.date()),
                "ticker": ticker,
            })

        snapshot_log.append(meta)

        time.sleep(0.25)

    membership = snapshots_to_membership(snapshot_rows)
    snapshot_log_df = pd.DataFrame(snapshot_log)

    ensure_dir("data")

    membership.to_csv(config.reconstructed_membership_file, index=False)
    snapshot_log_df.to_csv(config.snapshot_log_file, index=False)

    return membership, snapshot_log_df


def load_membership_schedule(config: BacktestConfig) -> Tuple[pd.DataFrame, str]:
    """
    Carga o reconstruye el membership histórico.

    Orden:
    1. Si existe data/sp100_membership.csv manual, lo usa.
    2. Si existe membership reconstruido y force_rebuild_membership=False, lo usa.
    3. Si no existe, reconstruye desde revisiones históricas de Wikipedia.
    4. Si todo falla y allow_current_fallback=False, lanza error.
    """
    manual_path = Path(config.membership_file)
    reconstructed_path = Path(config.reconstructed_membership_file)

    if manual_path.exists():
        membership = pd.read_csv(manual_path)

        required = {"ticker", "start", "end"}
        missing = required - set(membership.columns)

        if missing:
            raise ValueError(f"Faltan columnas en {manual_path}: {missing}")

        membership["ticker"] = membership["ticker"].astype(str).str.strip()
        membership["yf_ticker"] = membership["ticker"].map(to_yfinance_ticker)
        membership["start"] = pd.to_datetime(membership["start"], errors="coerce")
        membership["end"] = pd.to_datetime(membership["end"], errors="coerce")

        return membership, "manual_historical_membership_file"

    if (
        config.use_reconstructed_membership
        and reconstructed_path.exists()
        and not config.force_rebuild_membership
    ):
        membership = pd.read_csv(reconstructed_path)

        membership["ticker"] = membership["ticker"].astype(str).str.strip()
        membership["yf_ticker"] = membership["ticker"].map(to_yfinance_ticker)
        membership["start"] = pd.to_datetime(membership["start"], errors="coerce")
        membership["end"] = pd.to_datetime(membership["end"], errors="coerce")

        return membership, "cached_wikipedia_reconstructed_historical_membership"

    if config.use_reconstructed_membership:
        try:
            membership, snapshot_log_df = reconstruct_sp100_membership_from_wikipedia(config)
            return membership, "wikipedia_reconstructed_historical_membership"

        except Exception as e:
            if not config.allow_current_fallback:
                raise RuntimeError(
                    "No se pudo reconstruir la composición histórica del S&P 100 y "
                    "allow_current_fallback=False. No se usará lista actual para evitar survivorship bias. "
                    f"Error original: {e}"
                )

    if config.allow_current_fallback:
        tickers = sorted(SP100_FALLBACK_TICKERS)

        membership = pd.DataFrame({
            "ticker": tickers,
            "yf_ticker": [to_yfinance_ticker(t) for t in tickers],
            "start": pd.Timestamp("1900-01-01"),
            "end": pd.NaT,
            "source": "current_fallback",
        })

        return membership, "current_fallback_SURVIVORSHIP_BIASED"

    raise RuntimeError("No se pudo cargar ni reconstruir membership histórico.")


def active_tickers_on_date(membership: pd.DataFrame, date: pd.Timestamp) -> List[str]:
    """
    Devuelve los tickers activos en una fecha concreta según la tabla de pertenencia.
    """
    date = pd.Timestamp(date)

    active = membership[
        (membership["start"].isna() | (membership["start"] <= date)) &
        (membership["end"].isna() | (membership["end"] >= date))
    ]

    return sorted(active["yf_ticker"].dropna().unique().tolist())


def all_tickers_needed(membership: pd.DataFrame, start_date: str, end_date: str) -> List[str]:
    """
    Devuelve todos los tickers que podrían ser necesarios entre start_date y end_date.
    """
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    relevant = membership[
        (membership["end"].isna() | (membership["end"] >= start)) &
        (membership["start"].isna() | (membership["start"] <= end))
    ]

    return sorted(relevant["yf_ticker"].dropna().unique().tolist())


In [ ]:
# ============================================================
# 4. DESCARGA Y PREPROCESADO DE DATOS


In [ ]:
# ============================================================

def download_adjusted_close(
    tickers: Iterable[str],
    start_date: str,
    end_date: str,
    chunk_size: int = 25,
    sleep_seconds: float = 1.0,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Descarga precios ajustados mediante yfinance.

    Con auto_adjust=True, la columna Close queda ajustada.
    Descarga por bloques para reducir errores en Colab/Yahoo.
    """
    tickers = sorted(set([to_yfinance_ticker(t) for t in tickers]))

    if not tickers:
        raise ValueError("No hay tickers para descargar.")

    all_prices = []
    report_rows = []

    print(f"Descargando {len(tickers)} tickers desde Yahoo Finance...")

    for i in range(0, len(tickers), chunk_size):
        chunk = tickers[i:i + chunk_size]
        print(f"  Bloque {i // chunk_size + 1}: {len(chunk)} tickers")

        try:
            data = yf.download(
                tickers=chunk,
                start=start_date,
                end=end_date,
                auto_adjust=True,
                progress=False,
                group_by="ticker",
                threads=True,
            )

        except Exception as e:
            for t in chunk:
                report_rows.append({
                    "yf_ticker": t,
                    "status": f"download_exception: {type(e).__name__}",
                    "coverage": 0.0,
                    "first_valid_date": None,
                    "last_valid_date": None,
                })

            time.sleep(sleep_seconds)
            continue

        prices_chunk = pd.DataFrame()

        if len(chunk) == 1:
            t = chunk[0]

            if isinstance(data, pd.DataFrame) and "Close" in data.columns:
                prices_chunk[t] = data["Close"]

        else:
            for t in chunk:
                try:
                    if isinstance(data.columns, pd.MultiIndex):
                        if t in data.columns.get_level_values(0):
                            close = data[t]["Close"].copy()
                            prices_chunk[t] = close
                    else:
                        if "Close" in data.columns:
                            prices_chunk[t] = data["Close"]

                except Exception:
                    pass

        if not prices_chunk.empty:
            prices_chunk = prices_chunk.sort_index()
            prices_chunk.index.name = "Date"
            prices_chunk = prices_chunk.dropna(axis=1, how="all")
            all_prices.append(prices_chunk)

        available = set(prices_chunk.columns) if not prices_chunk.empty else set()

        for t in chunk:
            if t in available:
                s = prices_chunk[t]
                coverage = float(s.notna().mean())
                first_date = s.first_valid_index()
                last_date = s.last_valid_index()
                status = "ok"
            else:
                coverage = 0.0
                first_date = None
                last_date = None
                status = "download_failed_or_no_close"

            report_rows.append({
                "yf_ticker": t,
                "status": status,
                "coverage": coverage,
                "first_valid_date": first_date,
                "last_valid_date": last_date,
            })

        time.sleep(sleep_seconds)

    if not all_prices:
        raise RuntimeError("No se han podido descargar precios. Revisa conexión, tickers o yfinance.")

    prices = pd.concat(all_prices, axis=1)
    prices = prices.loc[:, ~prices.columns.duplicated()]
    prices = prices.sort_index()
    prices.index.name = "Date"

    report = pd.DataFrame(report_rows)

    return prices, report


def clean_prices(
    prices: pd.DataFrame,
    min_coverage: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Limpieza conservadora.
    """
    prices = prices.copy()
    prices = prices[~prices.index.duplicated(keep="first")]
    prices = prices.sort_index()

    coverage = prices.notna().mean()

    keep = coverage[coverage >= min_coverage].index.tolist()
    excluded = coverage[coverage < min_coverage].index.tolist()

    exclusions = pd.DataFrame({
        "yf_ticker": excluded,
        "reason": f"coverage_below_{min_coverage:.0%}",
        "coverage": coverage.loc[excluded].values if len(excluded) > 0 else [],
    })

    clean = prices[keep].copy()
    clean = clean.ffill()

    return clean, exclusions


def compute_simple_returns(prices: pd.DataFrame) -> pd.DataFrame:
    returns = prices.pct_change(fill_method=None)
    returns = returns.replace([np.inf, -np.inf], np.nan)

    return returns


In [ ]:
# ============================================================
# 5. BACKTESTS BASE


In [ ]:
# ============================================================

def backtest_static_buy_hold(
    prices: pd.DataFrame,
    membership: pd.DataFrame,
    config: BacktestConfig,
    transaction_cost: Optional[float] = None,
) -> Tuple[pd.Series, pd.Series, pd.DataFrame, dict]:
    """
    Buy & Hold estático:
    - Toma los miembros activos al inicio del backtest.
    - Compra cartera equally weighted.
    - No rebalancea.
    """
    transaction_cost = config.transaction_cost if transaction_cost is None else transaction_cost

    prices = prices.copy().dropna(axis=0, how="all")
    first_date = prices.index[0]

    active = active_tickers_on_date(membership, first_date)
    active = [t for t in active if t in prices.columns]

    first_prices = prices.loc[first_date, active].dropna()
    active = first_prices.index.tolist()

    if len(active) < config.min_assets_required:
        raise RuntimeError(
            f"Activos disponibles insuficientes para Buy & Hold: {len(active)} "
            f"(mínimo {config.min_assets_required})."
        )

    n = len(active)
    gross_weights = pd.Series(1.0 / n, index=active)

    entry_turnover = gross_weights.abs().sum()
    initial_cost = transaction_cost * entry_turnover * config.initial_capital
    investable_capital = config.initial_capital - initial_cost

    shares = (investable_capital * gross_weights) / first_prices

    portfolio_values = prices[active].ffill().mul(shares, axis=1).sum(axis=1)
    equity = portfolio_values.rename("strategy_equity")

    equity.iloc[0] = investable_capital

    returns = equity.pct_change(fill_method=None)
    returns.iloc[0] = equity.iloc[0] / config.initial_capital - 1.0
    returns = returns.rename("strategy_returns")

    weights = pd.DataFrame(0.0, index=prices.index, columns=active)
    weights.loc[:, active] = gross_weights.values

    metadata = {
        "strategy": "static_buy_hold",
        "first_date": str(first_date.date()),
        "n_assets": n,
        "transaction_cost": transaction_cost,
        "entry_turnover": float(entry_turnover),
        "initial_cost": float(initial_cost),
        "active_tickers": active,
    }

    return returns, equity, weights, metadata


def backtest_dynamic_equal_weight(
    prices: pd.DataFrame,
    membership: pd.DataFrame,
    config: BacktestConfig,
    transaction_cost: Optional[float] = None,
) -> Tuple[pd.Series, pd.Series, pd.DataFrame, dict]:
    """
    Equal-weight dinámico:
    - En cada fecha de rebalanceo selecciona miembros activos según membership histórico.
    - Rebalancea a pesos iguales.
    """
    transaction_cost = config.transaction_cost if transaction_cost is None else transaction_cost

    prices = prices.copy().dropna(axis=0, how="all").ffill()
    returns_assets = compute_simple_returns(prices).fillna(0.0)

    dates = prices.index

    rebalance_dates_raw = prices.resample(config.rebalance_frequency).last().index
    rebalance_dates = []

    for d in rebalance_dates_raw:
        idx = prices.index.get_indexer([d], method="pad")[0]

        if idx >= 0:
            rebalance_dates.append(prices.index[idx])

    rebalance_dates = sorted(set([d for d in rebalance_dates if d in dates]))

    all_cols = prices.columns.tolist()

    weights = pd.DataFrame(0.0, index=dates, columns=all_cols)

    current_weights = pd.Series(0.0, index=all_cols)
    prev_weights = pd.Series(0.0, index=all_cols)

    equity = pd.Series(index=dates, dtype=float, name="strategy_equity")
    strategy_returns = pd.Series(index=dates, dtype=float, name="strategy_returns")

    capital = config.initial_capital
    rebalance_count = 0
    total_turnover = 0.0

    for i, date in enumerate(dates):
        cost = 0.0

        if i == 0 or date in rebalance_dates:
            active = active_tickers_on_date(membership, date)
            active = [t for t in active if t in all_cols and not pd.isna(prices.loc[date, t])]

            if len(active) >= config.min_assets_required:
                target_weights = pd.Series(0.0, index=all_cols)
                target_weights.loc[active] = 1.0 / len(active)
            else:
                target_weights = prev_weights.copy()

            turnover = float((target_weights - prev_weights).abs().sum())
            cost = transaction_cost * turnover

            total_turnover += turnover
            rebalance_count += 1

            current_weights = target_weights.copy()
            prev_weights = current_weights.copy()

        if i == 0:
            r = -cost
        else:
            r_gross = float(current_weights.fillna(0.0).dot(returns_assets.iloc[i].fillna(0.0)))
            r = r_gross - cost

        capital *= (1.0 + r)

        equity.loc[date] = capital
        strategy_returns.loc[date] = r
        weights.loc[date] = current_weights.values

    metadata = {
        "strategy": "dynamic_equal_weight",
        "rebalance_frequency": config.rebalance_frequency,
        "transaction_cost": transaction_cost,
        "rebalance_count": rebalance_count,
        "total_turnover": total_turnover,
        "average_turnover_per_rebalance": total_turnover / rebalance_count if rebalance_count else np.nan,
    }

    return strategy_returns, equity, weights, metadata


def backtest_benchmark(
    benchmark_prices: pd.Series,
    config: BacktestConfig,
) -> Tuple[pd.Series, pd.Series]:
    """
    Benchmark Buy & Hold.
    No aplicamos costes al benchmark por defecto.
    """
    px = benchmark_prices.dropna().copy()
    px = px / px.iloc[0] * config.initial_capital
    px.name = "benchmark_equity"

    ret = px.pct_change(fill_method=None)
    ret.iloc[0] = 0.0
    ret.name = "benchmark_returns"

    return ret, px


In [ ]:
# ============================================================
# 6. MÉTRICAS


In [ ]:
# ============================================================

def format_index_endpoint(x):
    """
    Convierte un índice a string de forma robusta.
    """
    if hasattr(x, "date"):
        return str(x.date())

    return str(x)


def max_drawdown(equity: pd.Series) -> float:
    equity = equity.dropna()

    if len(equity) == 0:
        return np.nan

    running_max = equity.cummax()
    dd = equity / running_max - 1.0

    return float(dd.min())


def drawdown_series(equity: pd.Series) -> pd.Series:
    equity = equity.dropna()

    if len(equity) == 0:
        return pd.Series(dtype=float, name="drawdown")

    running_max = equity.cummax()

    return (equity / running_max - 1.0).rename("drawdown")


def annualized_volatility(
    returns: pd.Series,
    trading_days: int = 252,
) -> float:
    returns = returns.dropna()

    if len(returns) < 2:
        return np.nan

    return float(returns.std(ddof=1) * math.sqrt(trading_days))


def cagr_from_equity(
    equity: pd.Series,
    initial_capital: float,
    trading_days: int = 252,
) -> float:
    equity = equity.dropna()

    if len(equity) < 2:
        return np.nan

    years = len(equity) / trading_days
    final_value = equity.iloc[-1]

    if final_value <= 0:
        return -1.0

    return float((final_value / initial_capital) ** (1.0 / years) - 1.0)


def compute_metrics(
    returns: pd.Series,
    equity: pd.Series,
    config: BacktestConfig,
    label: str,
) -> pd.Series:
    returns = returns.dropna()
    equity = equity.dropna()

    if len(returns) == 0 or len(equity) == 0:
        return pd.Series({
            "label": label,
            "start": None,
            "end": None,
            "n_observations": 0,
            "cumulative_return": np.nan,
            "CAGR": np.nan,
            "annualized_volatility": np.nan,
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "max_drawdown": np.nan,
            "Calmar": np.nan,
            "hit_ratio": np.nan,
            "risk_free_rate_annual": config.risk_free_rate_annual,
        })

    rf_daily = (1.0 + config.risk_free_rate_annual) ** (1.0 / config.trading_days_per_year) - 1.0
    excess = returns - rf_daily

    cumulative_return = float(equity.iloc[-1] / config.initial_capital - 1.0)

    cagr = cagr_from_equity(
        equity,
        config.initial_capital,
        config.trading_days_per_year,
    )

    vol = annualized_volatility(
        returns,
        config.trading_days_per_year,
    )

    if len(excess) > 1 and excess.std(ddof=1) > 0:
        sharpe = float(
            excess.mean() / excess.std(ddof=1) * math.sqrt(config.trading_days_per_year)
        )
    else:
        sharpe = np.nan

    downside = excess[excess < 0]

    if len(downside) > 1 and downside.std(ddof=1) > 0:
        sortino = float(
            excess.mean() / downside.std(ddof=1) * math.sqrt(config.trading_days_per_year)
        )
    else:
        sortino = np.nan

    mdd = max_drawdown(equity)

    if not np.isnan(mdd) and mdd < 0:
        calmar = float(cagr / abs(mdd))
    else:
        calmar = np.nan

    hit_ratio = float((returns > 0).mean())

    return pd.Series({
        "label": label,
        "start": format_index_endpoint(equity.index[0]),
        "end": format_index_endpoint(equity.index[-1]),
        "n_observations": int(len(returns)),
        "cumulative_return": cumulative_return,
        "CAGR": cagr,
        "annualized_volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "max_drawdown": mdd,
        "Calmar": calmar,
        "hit_ratio": hit_ratio,
        "risk_free_rate_annual": config.risk_free_rate_annual,
    })


def compute_turnover(weights: pd.DataFrame) -> pd.Series:
    if weights.empty:
        return pd.Series(dtype=float, name="turnover")

    turnover = weights.diff().abs().sum(axis=1)
    turnover.iloc[0] = weights.iloc[0].abs().sum()

    return turnover.rename("turnover")


In [ ]:
# ============================================================
# 7. BOOTSTRAP SIMPLE


In [ ]:
# ============================================================

def bootstrap_metrics_simple(
    returns: pd.Series,
    config: BacktestConfig,
    n_bootstrap: int = 1000,
) -> pd.DataFrame:
    """
    Bootstrap simple sobre rentabilidades.
    Más adelante podemos sustituirlo por block bootstrap.
    """
    rng = np.random.default_rng(config.random_seed)
    r = returns.dropna().values

    rows = []

    for b in range(n_bootstrap):
        sample = rng.choice(r, size=len(r), replace=True)
        equity = pd.Series(config.initial_capital * np.cumprod(1.0 + sample))
        sample_returns = pd.Series(sample)

        m = compute_metrics(sample_returns, equity, config, label=f"bootstrap_{b}")
        rows.append(m)

    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# 8. FIGURAS


In [ ]:
# ============================================================

def plot_equity_curves(
    strategy_equity: pd.Series,
    benchmark_equity: pd.Series,
    output_path: str | Path,
) -> None:
    plt.figure(figsize=(10, 6))
    strategy_equity.dropna().plot(label="Strategy")
    benchmark_equity.dropna().plot(label="Benchmark")
    plt.title("Curva de capital")
    plt.xlabel("Fecha")
    plt.ylabel("Capital normalizado")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def plot_drawdowns(
    strategy_equity: pd.Series,
    benchmark_equity: pd.Series,
    output_path: str | Path,
) -> None:
    plt.figure(figsize=(10, 6))
    drawdown_series(strategy_equity).plot(label="Strategy")
    drawdown_series(benchmark_equity).plot(label="Benchmark")
    plt.title("Drawdown")
    plt.xlabel("Fecha")
    plt.ylabel("Drawdown")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


In [ ]:
# ============================================================
# 9. PIPELINE PRINCIPAL


In [ ]:
# ============================================================

def run_block0(config: BacktestConfig = CONFIG) -> None:
    warnings.filterwarnings("ignore", category=FutureWarning)
    np.random.seed(config.random_seed)

    out = ensure_dir(config.output_dir)
    ensure_dir(out / "figures")
    ensure_dir("data/raw")
    ensure_dir("data/processed")
    ensure_dir("logs")

    run_metadata = {
        "run_timestamp_utc": now_utc_iso(),
        "config": asdict(config),
    }

    save_json(run_metadata, out / "config_run.json")

    # Universo histórico
    membership, membership_source = load_membership_schedule(config)
    run_metadata["membership_source"] = membership_source

    tickers_needed = all_tickers_needed(
        membership,
        config.start_date,
        config.end_date,
    )

    all_download_tickers = sorted(set(tickers_needed + [config.benchmark_ticker]))

    # Descarga
    prices_raw, download_report = download_adjusted_close(
        all_download_tickers,
        config.start_date,
        config.end_date,
    )

    prices_raw.to_csv("data/raw/prices_raw.csv")
    download_report.to_csv("logs/download_report.csv", index=False)

    if config.benchmark_ticker not in prices_raw.columns:
        raise RuntimeError(
            f"No se pudo descargar el benchmark {config.benchmark_ticker}. "
            f"Prueba con '^GSPC' o revisa conexión/yfinance."
        )

    benchmark_prices = prices_raw[config.benchmark_ticker].dropna()

    asset_prices_raw = prices_raw.drop(columns=[config.benchmark_ticker], errors="ignore")
    asset_prices, exclusions = clean_prices(asset_prices_raw, config.min_price_coverage)

    asset_prices.to_csv("data/processed/prices_clean.csv")
    compute_simple_returns(asset_prices).to_csv("data/processed/returns.csv")
    exclusions.to_csv("logs/exclusions.csv", index=False)

    # Backtest de estrategia
    if config.strategy_mode == "static_buy_hold":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_static_buy_hold(
            asset_prices,
            membership,
            config,
        )

    elif config.strategy_mode == "dynamic_equal_weight":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_dynamic_equal_weight(
            asset_prices,
            membership,
            config,
        )

    else:
        raise ValueError(f"strategy_mode no reconocido: {config.strategy_mode}")

    # Benchmark
    benchmark_returns, benchmark_equity = backtest_benchmark(
        benchmark_prices,
        config,
    )

    # Alinear fechas
    common_dates = strategy_equity.index.intersection(benchmark_equity.index)

    strategy_equity = strategy_equity.loc[common_dates]
    strategy_returns = strategy_returns.loc[common_dates]

    benchmark_equity = benchmark_equity.loc[common_dates]
    benchmark_returns = benchmark_returns.loc[common_dates]

    # Métricas principales
    strategy_metrics = compute_metrics(
        strategy_returns,
        strategy_equity,
        config,
        "SP100_strategy",
    )

    benchmark_metrics = compute_metrics(
        benchmark_returns,
        benchmark_equity,
        config,
        config.benchmark_name,
    )

    metrics = pd.DataFrame([strategy_metrics, benchmark_metrics])

    # Turnover
    turnover = compute_turnover(weights)

    # Sensibilidad a costes
    sensitivity_rows = []

    for cost in config.cost_sensitivity:
        if config.strategy_mode == "static_buy_hold":
            r_s, eq_s, _, _ = backtest_static_buy_hold(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        else:
            r_s, eq_s, _, _ = backtest_dynamic_equal_weight(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        eq_s = eq_s.loc[eq_s.index.intersection(common_dates)]
        r_s = r_s.loc[eq_s.index]

        m = compute_metrics(r_s, eq_s, config, f"strategy_cost_{cost}")
        m["transaction_cost"] = cost
        sensitivity_rows.append(m)

    sensitivity = pd.DataFrame(sensitivity_rows)

    # Bootstrap simple
    boot = bootstrap_metrics_simple(
        strategy_returns,
        config,
        n_bootstrap=1000,
    )

    boot_summary = boot[["CAGR", "Sharpe", "Sortino", "max_drawdown", "Calmar"]].quantile(
        [0.025, 0.05, 0.50, 0.95, 0.975]
    )

    # Exportar outputs
    strategy_returns.to_csv(out / "strategy_returns.csv")
    strategy_equity.to_csv(out / "strategy_equity.csv")

    benchmark_returns.to_csv(out / "benchmark_returns.csv")
    benchmark_equity.to_csv(out / "benchmark_equity.csv")

    weights.to_csv(out / "weights.csv")
    turnover.to_csv(out / "turnover.csv")

    metrics.to_csv(out / "metrics.csv", index=False)
    sensitivity.to_csv(out / "cost_sensitivity.csv", index=False)

    boot.to_csv(out / "bootstrap_metrics.csv", index=False)
    boot_summary.to_csv(out / "bootstrap_summary.csv")

    # Metadata
    run_metadata["strategy_metadata"] = strategy_meta
    run_metadata["n_raw_assets_downloaded"] = len(tickers_needed)
    run_metadata["n_assets_after_cleaning"] = int(asset_prices.shape[1])
    run_metadata["n_excluded_assets"] = int(len(exclusions))
    run_metadata["benchmark_ticker"] = config.benchmark_ticker

    if "SURVIVORSHIP_BIASED" in membership_source:
        membership_warning = "WARNING: current/fallback constituents used. Survivorship bias present."
    elif "wikipedia" in membership_source.lower():
        membership_warning = (
            "Historical public reconstruction from Wikipedia page revisions. "
            "This avoids using future constituents, but is not the official licensed S&P DJI constituent history."
        )
    else:
        membership_warning = "Historical membership file used."

    run_metadata["membership_warning"] = membership_warning

    save_json(run_metadata, out / "metadata.json")

    # Figuras
    plot_equity_curves(
        strategy_equity,
        benchmark_equity,
        out / "figures/equity_curves.png",
    )

    plot_drawdowns(
        strategy_equity,
        benchmark_equity,
        out / "figures/drawdowns.png",
    )

    # Resumen por pantalla
    print("\n=== Bloque 0 completado ===")
    print(f"Modo estrategia: {config.strategy_mode}")
    print(f"Fuente membership: {membership_source}")
    print(f"Activos tras limpieza: {asset_prices.shape[1]}")
    print(f"Benchmark: {config.benchmark_ticker}")
    print(f"Outputs guardados en: {out.resolve()}")

    print("\nMétricas principales:")
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(metrics)


print("Bloque 0 cargado correctamente. Ahora cambia el Bloque 3 y ejecuta el Bloque 4.")


In [ ]:
# ============================================================
# BLOQUE 2.5 - CACHÉ DE MEMBERSHIP Y DATOS DE MERCADO


In [ ]:
# ============================================================

import hashlib


def make_market_data_cache_key(
    tickers: list,
    start_date: str,
    end_date: str,
    data_source: str,
) -> str:
    """
    Genera una clave única para identificar una descarga de datos.

    Si cambias:
    - tickers,
    - start_date,
    - end_date,
    - fuente de datos,

    se genera una caché distinta.
    """
    raw = {
        "tickers": sorted(tickers),
        "start_date": start_date,
        "end_date": end_date,
        "data_source": data_source,
    }

    raw_string = json.dumps(raw, sort_keys=True)
    return hashlib.md5(raw_string.encode("utf-8")).hexdigest()[:12]


def load_or_download_market_data(
    tickers: list,
    config: BacktestConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, str]:
    """
    Carga precios desde caché si existen.

    Si no existen o si CONFIG.force_redownload_market_data=True,
    descarga desde Yahoo Finance y guarda caché.
    """
    cache_dir = ensure_dir(getattr(config, "market_data_cache_dir", "data/cache"))

    cache_key = make_market_data_cache_key(
        tickers=tickers,
        start_date=config.start_date,
        end_date=config.end_date,
        data_source=config.data_source,
    )

    prices_cache_path = cache_dir / f"prices_{cache_key}.csv"
    report_cache_path = cache_dir / f"download_report_{cache_key}.csv"
    metadata_cache_path = cache_dir / f"market_data_metadata_{cache_key}.json"

    use_cached_market_data = getattr(config, "use_cached_market_data", True)
    force_redownload_market_data = getattr(config, "force_redownload_market_data", False)

    if (
        use_cached_market_data
        and not force_redownload_market_data
        and prices_cache_path.exists()
        and report_cache_path.exists()
    ):
        print(f"Cargando datos de mercado desde caché: {prices_cache_path}")

        prices = pd.read_csv(
            prices_cache_path,
            index_col=0,
            parse_dates=True,
        )

        prices.index.name = "Date"
        prices = prices.sort_index()

        report = pd.read_csv(report_cache_path)

        return prices, report, "market_data_cache"

    print("No hay caché válida de datos de mercado. Descargando desde Yahoo Finance...")

    prices, report = download_adjusted_close(
        tickers,
        config.start_date,
        config.end_date,
    )

    prices.to_csv(prices_cache_path)
    report.to_csv(report_cache_path, index=False)

    metadata = {
        "created_at_utc": now_utc_iso(),
        "cache_key": cache_key,
        "start_date": config.start_date,
        "end_date": config.end_date,
        "data_source": config.data_source,
        "n_tickers_requested": len(tickers),
        "tickers": sorted(tickers),
        "prices_cache_path": str(prices_cache_path),
        "report_cache_path": str(report_cache_path),
    }

    save_json(metadata, metadata_cache_path)

    print(f"Datos guardados en caché: {prices_cache_path}")

    return prices, report, "market_data_downloaded"


def run_block0(config: BacktestConfig = CONFIG) -> None:
    """
    Pipeline principal con caché.

    Cambios respecto al anterior:
    - Puede reutilizar membership histórico reconstruido.
    - Puede reutilizar precios descargados de Yahoo Finance.
    - Evita repetir los 6 minutos de descarga/reconstrucción cada vez.
    """
    warnings.filterwarnings("ignore", category=FutureWarning)
    np.random.seed(config.random_seed)

    out = ensure_dir(config.output_dir)
    ensure_dir(out / "figures")
    ensure_dir("data/raw")
    ensure_dir("data/processed")
    ensure_dir("logs")
    ensure_dir("data/cache")

    run_metadata = {
        "run_timestamp_utc": now_utc_iso(),
        "config": asdict(config),
    }

    # Guardamos también atributos añadidos dinámicamente en Colab.
    dynamic_config_attrs = [
        "use_cached_market_data",
        "force_redownload_market_data",
        "market_data_cache_dir",
    ]

    for attr in dynamic_config_attrs:
        if hasattr(config, attr):
            run_metadata["config"][attr] = getattr(config, attr)

    save_json(run_metadata, out / "config_run.json")

    # ========================================================
    # 1. Universo histórico
    # ========================================================

    membership, membership_source = load_membership_schedule(config)
    run_metadata["membership_source"] = membership_source

    tickers_needed = all_tickers_needed(
        membership,
        config.start_date,
        config.end_date,
    )

    all_download_tickers = sorted(set(tickers_needed + [config.benchmark_ticker]))

    # ========================================================
    # 2. Datos de mercado con caché
    # ========================================================

    prices_raw, download_report, market_data_source = load_or_download_market_data(
        all_download_tickers,
        config,
    )

    run_metadata["market_data_source"] = market_data_source

    prices_raw.to_csv("data/raw/prices_raw.csv")
    download_report.to_csv("logs/download_report.csv", index=False)

    if config.benchmark_ticker not in prices_raw.columns:
        raise RuntimeError(
            f"No se pudo descargar/cargar el benchmark {config.benchmark_ticker}. "
            f"Prueba con '^GSPC' o revisa conexión/yfinance."
        )

    benchmark_prices = prices_raw[config.benchmark_ticker].dropna()

    asset_prices_raw = prices_raw.drop(columns=[config.benchmark_ticker], errors="ignore")
    asset_prices, exclusions = clean_prices(asset_prices_raw, config.min_price_coverage)

    asset_prices.to_csv("data/processed/prices_clean.csv")
    compute_simple_returns(asset_prices).to_csv("data/processed/returns.csv")
    exclusions.to_csv("logs/exclusions.csv", index=False)

    # ========================================================
    # 3. Backtest de estrategia
    # ========================================================

    if config.strategy_mode == "static_buy_hold":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_static_buy_hold(
            asset_prices,
            membership,
            config,
        )

    elif config.strategy_mode == "dynamic_equal_weight":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_dynamic_equal_weight(
            asset_prices,
            membership,
            config,
        )

    else:
        raise ValueError(f"strategy_mode no reconocido: {config.strategy_mode}")

    # ========================================================
    # 4. Benchmark
    # ========================================================

    benchmark_returns, benchmark_equity = backtest_benchmark(
        benchmark_prices,
        config,
    )

    common_dates = strategy_equity.index.intersection(benchmark_equity.index)

    strategy_equity = strategy_equity.loc[common_dates]
    strategy_returns = strategy_returns.loc[common_dates]

    benchmark_equity = benchmark_equity.loc[common_dates]
    benchmark_returns = benchmark_returns.loc[common_dates]

    # ========================================================
    # 5. Métricas principales
    # ========================================================

    strategy_metrics = compute_metrics(
        strategy_returns,
        strategy_equity,
        config,
        "SP100_strategy",
    )

    benchmark_metrics = compute_metrics(
        benchmark_returns,
        benchmark_equity,
        config,
        config.benchmark_name,
    )

    metrics = pd.DataFrame([strategy_metrics, benchmark_metrics])

    # ========================================================
    # 6. Turnover
    # ========================================================

    turnover = compute_turnover(weights)

    # ========================================================
    # 7. Sensibilidad a costes
    # ========================================================

    sensitivity_rows = []

    for cost in config.cost_sensitivity:
        if config.strategy_mode == "static_buy_hold":
            r_s, eq_s, _, _ = backtest_static_buy_hold(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        else:
            r_s, eq_s, _, _ = backtest_dynamic_equal_weight(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        eq_s = eq_s.loc[eq_s.index.intersection(common_dates)]
        r_s = r_s.loc[eq_s.index]

        m = compute_metrics(r_s, eq_s, config, f"strategy_cost_{cost}")
        m["transaction_cost"] = cost
        sensitivity_rows.append(m)

    sensitivity = pd.DataFrame(sensitivity_rows)

    # ========================================================
    # 8. Bootstrap simple
    # ========================================================

    boot = bootstrap_metrics_simple(
        strategy_returns,
        config,
        n_bootstrap=1000,
    )

    boot_summary = boot[["CAGR", "Sharpe", "Sortino", "max_drawdown", "Calmar"]].quantile(
        [0.025, 0.05, 0.50, 0.95, 0.975]
    )

    # ========================================================
    # 9. Exportar outputs
    # ========================================================

    strategy_returns.to_csv(out / "strategy_returns.csv")
    strategy_equity.to_csv(out / "strategy_equity.csv")

    benchmark_returns.to_csv(out / "benchmark_returns.csv")
    benchmark_equity.to_csv(out / "benchmark_equity.csv")

    weights.to_csv(out / "weights.csv")
    turnover.to_csv(out / "turnover.csv")

    metrics.to_csv(out / "metrics.csv", index=False)
    sensitivity.to_csv(out / "cost_sensitivity.csv", index=False)

    boot.to_csv(out / "bootstrap_metrics.csv", index=False)
    boot_summary.to_csv(out / "bootstrap_summary.csv")

    # ========================================================
    # 10. Metadata
    # ========================================================

    run_metadata["strategy_metadata"] = strategy_meta
    run_metadata["n_raw_assets_downloaded"] = len(tickers_needed)
    run_metadata["n_assets_after_cleaning"] = int(asset_prices.shape[1])
    run_metadata["n_excluded_assets"] = int(len(exclusions))
    run_metadata["benchmark_ticker"] = config.benchmark_ticker

    if "SURVIVORSHIP_BIASED" in membership_source:
        membership_warning = "WARNING: current/fallback constituents used. Survivorship bias present."
    elif "wikipedia" in membership_source.lower():
        membership_warning = (
            "Historical public reconstruction from Wikipedia page revisions. "
            "This avoids using future constituents, but is not the official licensed S&P DJI constituent history."
        )
    else:
        membership_warning = "Historical membership file used."

    run_metadata["membership_warning"] = membership_warning

    save_json(run_metadata, out / "metadata.json")

    # ========================================================
    # 11. Figuras
    # ========================================================

    plot_equity_curves(
        strategy_equity,
        benchmark_equity,
        out / "figures/equity_curves.png",
    )

    plot_drawdowns(
        strategy_equity,
        benchmark_equity,
        out / "figures/drawdowns.png",
    )

    # ========================================================
    # 12. Resumen por pantalla
    # ========================================================

    print("\n=== Bloque 0 completado ===")
    print(f"Modo estrategia: {config.strategy_mode}")
    print(f"Fuente membership: {membership_source}")
    print(f"Fuente datos mercado: {market_data_source}")
    print(f"Activos tras limpieza: {asset_prices.shape[1]}")
    print(f"Benchmark: {config.benchmark_ticker}")
    print(f"Outputs guardados en: {out.resolve()}")

    print("\nMétricas principales:")
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(metrics)


print("Bloque 2.5 cargado correctamente. run_block0 ahora usa caché de datos.")

"""Configuración"""


In [ ]:
# ============================================================
# cELDA 3 - CONFIGURACIÓN DEL EXPERIMENTO


In [ ]:
# ============================================================

from pathlib import Path

# Asegurar carpetas en Drive
Path(f"{TFG_BASE_DIR}").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data/cache").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/results").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/logs").mkdir(parents=True, exist_ok=True)

CONFIG.start_date = "2021-01-01"
CONFIG.end_date = "2026-01-01"

# Benchmark S&P 500.
CONFIG.benchmark_ticker = "SPY"

# Usamos estrategia dinámica para respetar entradas/salidas del universo histórico.
CONFIG.strategy_mode = "dynamic_equal_weight"
CONFIG.rebalance_frequency = "M"

# Costes
CONFIG.transaction_cost = 0.001

# Tasa libre de riesgo anual provisional.
CONFIG.risk_free_rate_annual = 0.0225

# Reconstrucción histórica del S&P 100.
CONFIG.use_reconstructed_membership = True

# False = no reconstruye desde Wikipedia si ya existe el CSV reconstruido.
CONFIG.force_rebuild_membership = False

CONFIG.membership_history_frequency = "MS"

# Si falla la reconstrucción histórica, NO vuelve a lista actual.
CONFIG.allow_current_fallback = False

# Caché de precios.
CONFIG.use_cached_market_data = True

# False = no vuelve a descargar precios si ya existen en caché.
CONFIG.force_redownload_market_data = False

# Guardamos caché en Drive.
CONFIG.market_data_cache_dir = f"{TFG_BASE_DIR}/data/cache"

# Guardamos resultados en Drive.
CONFIG.output_dir = f"{TFG_BASE_DIR}/results/block0_sp100_survivorship_corrected"

# Guardamos membership histórico y logs en Drive.
CONFIG.reconstructed_membership_file = f"{TFG_BASE_DIR}/data/sp100_membership_reconstructed_wikipedia.csv"
CONFIG.snapshot_log_file = f"{TFG_BASE_DIR}/data/sp100_wikipedia_snapshot_log.csv"

CONFIG

"""Ejecutar"""

run_block0(CONFIG)

"""Resultados"""


In [ ]:
# ============================================================
# CELDA 5 - VER RESULTADOS PRINCIPALES


In [ ]:
# ============================================================

from pathlib import Path
import pandas as pd
import json

results_dir = Path(CONFIG.output_dir)

metrics_path = results_dir / "metrics.csv"
sensitivity_path = results_dir / "cost_sensitivity.csv"
bootstrap_summary_path = results_dir / "bootstrap_summary.csv"
metadata_path = results_dir / "metadata.json"

print(f"Carpeta de resultados: {results_dir}")

if not metrics_path.exists():
    raise FileNotFoundError(
        f"No existe {metrics_path}. Ejecuta primero el Bloque 4: run_block0(CONFIG)"
    )

metrics = pd.read_csv(metrics_path)
cost_sensitivity = pd.read_csv(sensitivity_path)
bootstrap_summary = pd.read_csv(bootstrap_summary_path, index_col=0)

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("\n=== Metadata clave ===")
print("Fuente membership:", metadata.get("membership_source"))
print("Aviso membership:", metadata.get("membership_warning"))
print("Fuente datos mercado:", metadata.get("market_data_source"))
print("Benchmark:", metadata.get("benchmark_ticker"))
print("Activos tras limpieza:", metadata.get("n_assets_after_cleaning"))

print("\n=== Métricas principales ===")
display(metrics)

print("\n=== Sensibilidad a costes ===")
display(cost_sensitivity)

print("\n=== Bootstrap summary ===")
display(bootstrap_summary)

"""Figuras"""


In [ ]:
# ============================================================
# CELDA 6 - VER FIGURAS


In [ ]:
# ============================================================

from pathlib import Path
from IPython.display import Image, display

results_dir = Path(CONFIG.output_dir)
figures_dir = results_dir / "figures"

equity_path = figures_dir / "equity_curves.png"
drawdown_path = figures_dir / "drawdowns.png"

print(f"Carpeta de figuras: {figures_dir}")

if not equity_path.exists():
    raise FileNotFoundError(
        f"No existe {equity_path}. Ejecuta primero el Bloque 4: run_block0(CONFIG)"
    )

if not drawdown_path.exists():
    raise FileNotFoundError(
        f"No existe {drawdown_path}. Ejecuta primero el Bloque 4: run_block0(CONFIG)"
    )

print("\n=== Curva de capital ===")
display(Image(filename=str(equity_path)))

print("\n=== Drawdown ===")
display(Image(filename=str(drawdown_path)))

"""Descargar"""

!zip -r block0_results.zip results data logs > /dev/null

from google.colab import files
files.download("block0_results.zip")


In [ ]:
# ============================================================
# MÓDULO C3 - RÉPLICA OPERATIVA INSPIRADA EN UMA
# Celda C3.1 - Configuración y carga de datos


In [ ]:
# ============================================================

from pathlib import Path
import json
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

if "TFG_BASE_DIR" not in globals():
    raise RuntimeError("No encuentro TFG_BASE_DIR. Ejecuta primero el Módulo C0.")

if "CONFIG" not in globals():
    raise RuntimeError("No encuentro CONFIG. Ejecuta primero el Módulo C0.")

C3_OUTPUT_DIR = Path(TFG_BASE_DIR) / "results" / "C3_uma_replication"
C3_TABLES_DIR = C3_OUTPUT_DIR / "tables"
C3_FIGURES_DIR = C3_OUTPUT_DIR / "figures"

C3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
C3_TABLES_DIR.mkdir(parents=True, exist_ok=True)
C3_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Periodos del experimento.
# Entrenamiento: se usa para elegir la mejor variante.
# Evaluación: se usa para medir fuera de muestra y hacer White's Reality Check.
C3_TRAIN_END = "2023-12-31"
C3_TEST_START = "2024-01-01"

# Parámetros del experimento.
C3_TRANSACTION_COST = CONFIG.transaction_cost
C3_RISK_FREE_RATE_ANNUAL = CONFIG.risk_free_rate_annual
C3_TRADING_DAYS = CONFIG.trading_days_per_year
C3_INITIAL_CAPITAL = CONFIG.initial_capital

# Bootstrap y White's Reality Check.
C3_N_BOOTSTRAP = 1000
C3_WRC_BOOTSTRAP = 1000
C3_BLOCK_LENGTH = 20
C3_RANDOM_SEED = CONFIG.random_seed

# Criterio de selección de la estrategia en entrenamiento.
# Usamos Sortino porque encaja con la motivación de la tesis UMA.
C3_SELECTION_METRIC = "Sortino"

print("Carpeta de salida C3:", C3_OUTPUT_DIR)
print("Entrenamiento hasta:", C3_TRAIN_END)
print("Evaluación desde:", C3_TEST_START)
print("Coste proporcional:", C3_TRANSACTION_COST)


def c3_read_dataframe(path):
    path = Path(path)
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df = df.sort_index()
    return df


def c3_read_series(path, name=None):
    path = Path(path)
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df = df.sort_index()

    if isinstance(df, pd.Series):
        s = df
    else:
        s = df.iloc[:, 0]

    if name is not None:
        s.name = name

    return s


def c3_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


def c3_load_prices_from_cache():
    cache_dir = Path(getattr(CONFIG, "market_data_cache_dir", Path(TFG_BASE_DIR) / "data" / "cache"))

    if not cache_dir.exists():
        return None

    cache_files = sorted(
        cache_dir.glob("prices_*.csv"),
        key=lambda x: x.stat().st_mtime,
        reverse=True
    )

    for path in cache_files:
        try:
            raw = c3_read_dataframe(path)

            if CONFIG.benchmark_ticker in raw.columns:
                raw_assets = raw.drop(columns=[CONFIG.benchmark_ticker], errors="ignore")
            else:
                raw_assets = raw.copy()

            if "clean_prices" in globals():
                cleaned, _ = clean_prices(raw_assets, CONFIG.min_price_coverage)
            else:
                cleaned = raw_assets.replace([np.inf, -np.inf], np.nan).ffill()
                coverage = cleaned.notna().mean()
                cleaned = cleaned.loc[:, coverage >= CONFIG.min_price_coverage]
                cleaned = cleaned.dropna(axis=0, how="all").ffill()

            if cleaned.shape[1] >= CONFIG.min_assets_required:
                print("Precios cargados desde caché:", path)
                return cleaned

        except Exception as e:
            print("No se pudo usar caché:", path, "|", e)

    return None


# 1) Cargar precios limpios.
prices_path = c3_first_existing([
    Path(TFG_BASE_DIR) / "data" / "common_C0" / "prices_clean.csv",
    Path("data") / "processed" / "prices_clean.csv",
])

if prices_path is not None:
    prices_clean = c3_read_dataframe(prices_path)
    print("Precios cargados desde:", prices_path)
else:
    prices_clean = c3_load_prices_from_cache()

if prices_clean is None or prices_clean.empty:
    raise FileNotFoundError(
        "No he podido cargar prices_clean. Ejecuta antes el Módulo C0 completo."
    )

# 2) Cargar membership histórico.
membership_path = c3_first_existing([
    Path(getattr(CONFIG, "reconstructed_membership_file", "")),
    Path(TFG_BASE_DIR) / "data" / "sp100_membership_reconstructed_wikipedia.csv",
    Path("data") / "sp100_membership_reconstructed_wikipedia.csv",
])

if membership_path is not None:
    membership = pd.read_csv(membership_path)
    print("Membership cargado desde:", membership_path)
elif "load_membership_schedule" in globals():
    membership, membership_source = load_membership_schedule(CONFIG)
    print("Membership cargado usando load_membership_schedule:", membership_source)
else:
    raise FileNotFoundError("No he podido cargar el membership histórico del S&P 100.")

# 3) Cargar benchmark y Buy&Hold del Módulo C0.
c0_results_dir = Path(CONFIG.output_dir)

benchmark_returns_path = c0_results_dir / "benchmark_returns.csv"
benchmark_equity_path = c0_results_dir / "benchmark_equity.csv"
buyhold_returns_path = c0_results_dir / "strategy_returns.csv"
buyhold_equity_path = c0_results_dir / "strategy_equity.csv"

if not benchmark_returns_path.exists():
    raise FileNotFoundError(f"No existe {benchmark_returns_path}. Ejecuta primero C0.")

if not buyhold_returns_path.exists():
    raise FileNotFoundError(f"No existe {buyhold_returns_path}. Ejecuta primero C0.")

benchmark_returns = c3_read_series(benchmark_returns_path, name="Benchmark")
benchmark_equity = c3_read_series(benchmark_equity_path, name="Benchmark")
buyhold_returns = c3_read_series(buyhold_returns_path, name="Buy&Hold")
buyhold_equity = c3_read_series(buyhold_equity_path, name="Buy&Hold")

# 4) Recortar y alinear fechas.
prices_clean = prices_clean.loc[
    (prices_clean.index >= pd.Timestamp(CONFIG.start_date)) &
    (prices_clean.index < pd.Timestamp(CONFIG.end_date))
].copy()

common_dates = prices_clean.index
common_dates = common_dates.intersection(benchmark_returns.index)
common_dates = common_dates.intersection(buyhold_returns.index)

prices_clean = prices_clean.loc[common_dates]
benchmark_returns = benchmark_returns.loc[common_dates].fillna(0.0)
benchmark_equity = benchmark_equity.loc[benchmark_equity.index.intersection(common_dates)]
buyhold_returns = buyhold_returns.loc[common_dates].fillna(0.0)
buyhold_equity = buyhold_equity.loc[buyhold_equity.index.intersection(common_dates)]

print("Fechas:", prices_clean.index.min().date(), "a", prices_clean.index.max().date())
print("Activos disponibles:", prices_clean.shape[1])
print("Observaciones:", prices_clean.shape[0])


In [ ]:
# ============================================================
# Celda C3.2 - Funciones auxiliares, métricas, bootstrap y WRC


In [ ]:
# ============================================================

def c3_simple_returns(prices):
    return prices.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)


def c3_drawdown_series(equity):
    equity = equity.dropna()
    running_max = equity.cummax()
    return equity / running_max - 1.0


def c3_max_drawdown(equity):
    dd = c3_drawdown_series(equity)
    if dd.empty:
        return np.nan
    return float(dd.min())


def c3_equity_from_returns(returns, initial_capital=C3_INITIAL_CAPITAL):
    returns = returns.fillna(0.0)
    return initial_capital * (1.0 + returns).cumprod()


def c3_cagr(equity, trading_days=C3_TRADING_DAYS):
    equity = equity.dropna()

    if len(equity) < 2:
        return np.nan

    total_return = equity.iloc[-1] / equity.iloc[0]
    years = len(equity) / trading_days

    if years <= 0 or total_return <= 0:
        return np.nan

    return float(total_return ** (1.0 / years) - 1.0)


def c3_compute_metrics(returns, label):
    returns = returns.dropna().copy()
    equity = c3_equity_from_returns(returns)

    if len(returns) < 2:
        return pd.Series({
            "label": label,
            "start": None,
            "end": None,
            "n_observations": len(returns),
            "cumulative_return": np.nan,
            "CAGR": np.nan,
            "annualized_volatility": np.nan,
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "max_drawdown": np.nan,
            "Calmar": np.nan,
            "hit_ratio": np.nan,
        })

    rf_daily = (1.0 + C3_RISK_FREE_RATE_ANNUAL) ** (1.0 / C3_TRADING_DAYS) - 1.0
    excess = returns - rf_daily

    cumulative_return = float(equity.iloc[-1] / C3_INITIAL_CAPITAL - 1.0)
    cagr = c3_cagr(equity)
    vol = float(returns.std(ddof=1) * np.sqrt(C3_TRADING_DAYS))

    if excess.std(ddof=1) > 0:
        sharpe = float(excess.mean() / excess.std(ddof=1) * np.sqrt(C3_TRADING_DAYS))
    else:
        sharpe = np.nan

    downside = excess[excess < 0]

    if len(downside) > 1 and downside.std(ddof=1) > 0:
        sortino = float(excess.mean() / downside.std(ddof=1) * np.sqrt(C3_TRADING_DAYS))
    else:
        sortino = np.nan

    mdd = c3_max_drawdown(equity)

    if pd.notna(mdd) and mdd < 0:
        calmar = float(cagr / abs(mdd))
    else:
        calmar = np.nan

    hit_ratio = float((returns > 0).mean())

    return pd.Series({
        "label": label,
        "start": str(returns.index.min().date()) if hasattr(returns.index.min(), "date") else str(returns.index.min()),
        "end": str(returns.index.max().date()) if hasattr(returns.index.max(), "date") else str(returns.index.max()),
        "n_observations": int(len(returns)),
        "cumulative_return": cumulative_return,
        "CAGR": cagr,
        "annualized_volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "max_drawdown": mdd,
        "Calmar": calmar,
        "hit_ratio": hit_ratio,
    })


def c3_active_tickers_on_date_local(membership_df, date):
    date = pd.Timestamp(date)

    if "active_tickers_on_date" in globals():
        return active_tickers_on_date(membership_df, date)

    m = membership_df.copy()

    if "start" in m.columns and "end" in m.columns:
        m["start"] = pd.to_datetime(m["start"])
        m["end"] = pd.to_datetime(m["end"], errors="coerce")

        active = m[
            (m["start"] <= date) &
            (m["end"].isna() | (m["end"] >= date))
        ]

        ticker_col = "yf_ticker" if "yf_ticker" in active.columns else "ticker"
        return sorted(active[ticker_col].dropna().astype(str).unique().tolist())

    if "snapshot_date" in m.columns and "ticker" in m.columns:
        m["snapshot_date"] = pd.to_datetime(m["snapshot_date"])
        valid_dates = m[m["snapshot_date"] <= date]["snapshot_date"]

        if valid_dates.empty:
            return []

        last_date = valid_dates.max()
        return sorted(
            m.loc[m["snapshot_date"] == last_date, "ticker"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

    if "ticker" in m.columns:
        return sorted(m["ticker"].dropna().astype(str).unique().tolist())

    raise ValueError("Formato de membership no reconocido.")


def c3_build_active_mask(prices, membership_df):
    mask = pd.DataFrame(False, index=prices.index, columns=prices.columns)

    for date in prices.index:
        active = c3_active_tickers_on_date_local(membership_df, date)
        active = [t for t in active if t in prices.columns]
        if active:
            mask.loc[date, active] = True

    return mask


def c3_circular_block_indices(n, block_length, rng):
    indices = []

    while len(indices) < n:
        start = rng.integers(0, n)
        block = [(start + i) % n for i in range(block_length)]
        indices.extend(block)

    return np.array(indices[:n])


def c3_block_bootstrap_metrics(returns, label, n_bootstrap=C3_N_BOOTSTRAP, block_length=C3_BLOCK_LENGTH):
    rng = np.random.default_rng(C3_RANDOM_SEED)
    r = returns.dropna().values
    n = len(r)

    if n < 5:
        raise ValueError("Muy pocas observaciones para bootstrap.")

    rows = []

    for b in range(n_bootstrap):
        idx = c3_circular_block_indices(n, block_length, rng)
        sample = pd.Series(r[idx], index=range(n))
        metrics = c3_compute_metrics(sample, f"{label}_bootstrap_{b}")
        rows.append(metrics)

    boot = pd.DataFrame(rows)

    metrics_cols = [
        "cumulative_return",
        "CAGR",
        "annualized_volatility",
        "Sharpe",
        "Sortino",
        "max_drawdown",
        "Calmar",
        "hit_ratio",
    ]

    percentiles = boot[metrics_cols].quantile([0.025, 0.05, 0.50, 0.95, 0.975])
    percentiles.index = ["p2_5", "p5", "p50", "p95", "p97_5"]
    percentiles.insert(0, "label", label)

    return boot, percentiles


def c3_white_reality_check(candidate_returns, reference_returns, n_bootstrap=C3_WRC_BOOTSTRAP, block_length=C3_BLOCK_LENGTH):
    """
    White's Reality Check simplificado.

    H0: ninguna estrategia candidata supera a la referencia en rentabilidad media.
    Estadístico: máximo de la media diaria diferencial frente a la referencia.
    Bootstrap: bloque circular sobre diferenciales centrados.
    """

    data = candidate_returns.copy()
    ref = reference_returns.copy()

    common = data.index.intersection(ref.index)
    data = data.loc[common].dropna(axis=1, how="all").fillna(0.0)
    ref = ref.loc[common].fillna(0.0)

    diff = data.sub(ref, axis=0)
    diff = diff.dropna(axis=0, how="all")

    if diff.empty:
        raise ValueError("No hay datos para White's Reality Check.")

    mean_diff = diff.mean(axis=0)
    best_candidate = mean_diff.idxmax()
    observed_stat = float(mean_diff.max())

    centered = diff - diff.mean(axis=0)

    rng = np.random.default_rng(C3_RANDOM_SEED)
    n = len(centered)
    boot_stats = []

    centered_values = centered.values

    for b in range(n_bootstrap):
        idx = c3_circular_block_indices(n, block_length, rng)
        sample = centered_values[idx, :]
        sample_means = sample.mean(axis=0)
        boot_stats.append(float(np.max(sample_means)))

    boot_stats = np.array(boot_stats)
    p_value = float(np.mean(boot_stats >= observed_stat))

    summary = pd.DataFrame([{
        "best_candidate": best_candidate,
        "observed_mean_daily_excess_return": observed_stat,
        "observed_annualized_excess_return": observed_stat * C3_TRADING_DAYS,
        "p_value": p_value,
        "n_observations": int(n),
        "n_candidates": int(diff.shape[1]),
        "n_bootstrap": int(n_bootstrap),
        "block_length": int(block_length),
    }])

    candidate_table = pd.DataFrame({
        "candidate": mean_diff.index,
        "mean_daily_excess_return": mean_diff.values,
        "annualized_excess_return": mean_diff.values * C3_TRADING_DAYS,
    }).sort_values("mean_daily_excess_return", ascending=False)

    boot_table = pd.DataFrame({"bootstrap_max_mean": boot_stats})

    return summary, candidate_table, boot_table


In [ ]:
# ============================================================
# Celda C3.2b - Corrección robusta del membership histórico


In [ ]:
# ============================================================

def c3_normalize_membership(membership_df):
    """
    Normaliza el dataframe de composición histórica para evitar errores
    por columnas de fecha guardadas como texto.
    """
    m = membership_df.copy()

    # Limpiar nombres de columnas
    m.columns = [str(c).strip() for c in m.columns]

    # Detectar columnas típicas de ticker
    if "yf_ticker" in m.columns:
        ticker_col = "yf_ticker"
    elif "ticker" in m.columns:
        ticker_col = "ticker"
    elif "Ticker" in m.columns:
        ticker_col = "Ticker"
    else:
        raise ValueError(f"No encuentro columna de ticker en membership. Columnas: {m.columns.tolist()}")

    # Detectar formato de intervalos start/end
    possible_start_cols = ["start", "Start", "start_date", "Start Date", "from", "From"]
    possible_end_cols = ["end", "End", "end_date", "End Date", "to", "To"]

    start_col = next((c for c in possible_start_cols if c in m.columns), None)
    end_col = next((c for c in possible_end_cols if c in m.columns), None)

    if start_col is not None and end_col is not None:
        m[start_col] = pd.to_datetime(m[start_col], errors="coerce")
        m[end_col] = pd.to_datetime(m[end_col], errors="coerce")

        return {
            "type": "interval",
            "df": m,
            "ticker_col": ticker_col,
            "start_col": start_col,
            "end_col": end_col,
        }

    # Detectar formato snapshot_date
    possible_snapshot_cols = ["snapshot_date", "Snapshot Date", "date", "Date"]
    snapshot_col = next((c for c in possible_snapshot_cols if c in m.columns), None)

    if snapshot_col is not None:
        m[snapshot_col] = pd.to_datetime(m[snapshot_col], errors="coerce")

        return {
            "type": "snapshot",
            "df": m,
            "ticker_col": ticker_col,
            "snapshot_col": snapshot_col,
        }

    # Último recurso: lista estática de tickers
    return {
        "type": "static",
        "df": m,
        "ticker_col": ticker_col,
    }


C3_MEMBERSHIP_INFO = c3_normalize_membership(membership)

print("Membership normalizado.")
print("Tipo:", C3_MEMBERSHIP_INFO["type"])
print("Columnas:", C3_MEMBERSHIP_INFO["df"].columns.tolist())


def c3_active_tickers_on_date_local(membership_df, date):
    """
    Versión C3 independiente de la función del C0.
    No usa active_tickers_on_date global para evitar errores de fechas como string.
    """
    info = C3_MEMBERSHIP_INFO
    m = info["df"]
    ticker_col = info["ticker_col"]
    date = pd.Timestamp(date)

    if info["type"] == "interval":
        start_col = info["start_col"]
        end_col = info["end_col"]

        active = m[
            (m[start_col] <= date) &
            (m[end_col].isna() | (m[end_col] >= date))
        ]

        return sorted(
            active[ticker_col]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

    if info["type"] == "snapshot":
        snapshot_col = info["snapshot_col"]

        valid_dates = m.loc[m[snapshot_col] <= date, snapshot_col]

        if valid_dates.empty:
            return []

        last_date = valid_dates.max()

        active = m.loc[m[snapshot_col] == last_date]

        return sorted(
            active[ticker_col]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

    if info["type"] == "static":
        return sorted(
            m[ticker_col]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

    raise ValueError("Tipo de membership no reconocido.")


def c3_build_active_mask(prices, membership_df):
    mask = pd.DataFrame(False, index=prices.index, columns=prices.columns)

    for i, date in enumerate(prices.index):
        if i % 250 == 0:
            print(f"Construyendo active mask: {i}/{len(prices.index)}")

        active = c3_active_tickers_on_date_local(membership_df, date)
        active = [t for t in active if t in prices.columns]

        if active:
            mask.loc[date, active] = True

    return mask


In [ ]:
# ============================================================
# Celda C3.3 - GA cross-sectional con validación interna
# Objetivo: mejorar generalización fuera de muestra


In [ ]:
# ============================================================

asset_returns = c3_simple_returns(prices_clean).fillna(0.0)
active_mask = c3_build_active_mask(prices_clean, membership)

# División temporal:
# 2021-2022: entrenamiento interno
# 2023: validación interna
# 2024-2025: test real fuera de muestra
C3_INNER_TRAIN_END = "2022-12-31"
C3_VALIDATION_START = "2023-01-01"
C3_VALIDATION_END = "2023-12-31"

inner_train_mask = prices_clean.index <= pd.Timestamp(C3_INNER_TRAIN_END)

validation_mask = (
    (prices_clean.index >= pd.Timestamp(C3_VALIDATION_START)) &
    (prices_clean.index <= pd.Timestamp(C3_VALIDATION_END))
)

train_mask = prices_clean.index <= pd.Timestamp(C3_TRAIN_END)
test_mask = prices_clean.index >= pd.Timestamp(C3_TEST_START)
full_mask = prices_clean.index >= prices_clean.index.min()

# ------------------------------------------------------------
# Configuración del GA
# ------------------------------------------------------------

# El GA NO mira el test. Optimiza usando entrenamiento interno + validación interna.
C3_GA_OPTIMIZATION_SCOPE = "train_validation"

# Permitimos algo de apalancamiento, pero moderado.
C3_ALLOW_LEVERAGE = True
C3_MAX_LEVERAGE = 1.10 if C3_ALLOW_LEVERAGE else 1.00

# Ensemble de mejores estrategias para reducir sobreajuste.
C3_USE_ENSEMBLE = True
C3_ENSEMBLE_SIZE = 10

# Parámetros del algoritmo genético
C3_GA_POPULATION_SIZE = 90
C3_GA_GENERATIONS = 15
C3_GA_ELITE_SIZE = 8
C3_GA_TOURNAMENT_SIZE = 4
C3_GA_CROSSOVER_PROB = 0.90
C3_GA_MUTATION_PROB = 0.22
C3_GA_MUTATION_SCALE = 0.15

# Guardamos muchos candidatos para que White's Reality Check corrija
# por haber probado muchas alternativas.
C3_GA_KEEP_TOP = 300
C3_GA_RANDOM_SEED = C3_RANDOM_SEED

rng = np.random.default_rng(C3_GA_RANDOM_SEED)

print("Active mask construido.")
print("Fechas:", active_mask.shape[0])
print("Activos:", active_mask.shape[1])
print("Observaciones inner train:", int(inner_train_mask.sum()))
print("Observaciones validación:", int(validation_mask.sum()))
print("Observaciones train total:", int(train_mask.sum()))
print("Observaciones test:", int(test_mask.sum()))
print("Modo optimización:", C3_GA_OPTIMIZATION_SCOPE)
print("Apalancamiento permitido:", C3_ALLOW_LEVERAGE, "| Máximo:", C3_MAX_LEVERAGE)
print("Usar ensemble:", C3_USE_ENSEMBLE, "| Tamaño:", C3_ENSEMBLE_SIZE)

# ------------------------------------------------------------
# Cromosoma
# ------------------------------------------------------------
# Estrategia cross-sectional:
# cada rebalanceo selecciona los mejores activos según una combinación
# de momentum, baja volatilidad, RSI y ruptura de máximos.

C3_GENE_NAMES = [
    "mom_short_lb",
    "mom_mid_lb",
    "mom_long_lb",
    "vol_lb",
    "rsi_lb",
    "breakout_lb",
    "w_mom_short",
    "w_mom_mid",
    "w_mom_long",
    "w_low_vol",
    "w_rsi",
    "w_breakout",
    "top_n",
    "rebalance_days",
    "market_filter_lb",
    "leverage",
]

C3_BOUNDS_LOW = np.array([
    10,    # mom_short_lb
    40,    # mom_mid_lb
    90,    # mom_long_lb
    20,    # vol_lb
    7,     # rsi_lb
    30,    # breakout_lb
    -2.0,  # w_mom_short
    -2.0,  # w_mom_mid
    -2.0,  # w_mom_long
    -2.0,  # w_low_vol
    -2.0,  # w_rsi
    -2.0,  # w_breakout
    3,     # top_n
    5,     # rebalance_days
    20,    # market_filter_lb
    0.50,  # leverage
], dtype=float)

C3_BOUNDS_HIGH = np.array([
    80,              # mom_short_lb
    160,             # mom_mid_lb
    300,             # mom_long_lb
    120,             # vol_lb
    35,              # rsi_lb
    250,             # breakout_lb
    2.0,             # w_mom_short
    2.0,             # w_mom_mid
    2.0,             # w_mom_long
    2.0,             # w_low_vol
    2.0,             # w_rsi
    2.0,             # w_breakout
    35,              # top_n
    63,              # rebalance_days
    250,             # market_filter_lb
    C3_MAX_LEVERAGE, # leverage
], dtype=float)


def c3_ga_random_chromosome(rng):
    return rng.uniform(C3_BOUNDS_LOW, C3_BOUNDS_HIGH)


def c3_ga_repair(chromosome):
    x = np.array(chromosome, dtype=float).copy()
    x = np.clip(x, C3_BOUNDS_LOW, C3_BOUNDS_HIGH)

    # Ordenar lookbacks de momentum
    lbs = sorted([x[0], x[1], x[2]])
    x[0], x[1], x[2] = lbs[0], lbs[1], lbs[2]

    # Evitar todos los pesos casi cero
    if np.sum(np.abs(x[6:12])) < 0.25:
        x[6] = 1.0
        x[7] = 1.0
        x[8] = 1.0
        x[9] = 0.5
        x[10] = 0.5
        x[11] = 0.5

    if not C3_ALLOW_LEVERAGE:
        x[15] = min(x[15], 1.0)

    return np.clip(x, C3_BOUNDS_LOW, C3_BOUNDS_HIGH)


def c3_ga_decode(chromosome):
    x = c3_ga_repair(chromosome)

    params = {
        "mom_short_lb": int(round(x[0])),
        "mom_mid_lb": int(round(x[1])),
        "mom_long_lb": int(round(x[2])),
        "vol_lb": int(round(x[3])),
        "rsi_lb": int(round(x[4])),
        "breakout_lb": int(round(x[5])),
        "w_mom_short": float(x[6]),
        "w_mom_mid": float(x[7]),
        "w_mom_long": float(x[8]),
        "w_low_vol": float(x[9]),
        "w_rsi": float(x[10]),
        "w_breakout": float(x[11]),
        "top_n": int(round(x[12])),
        "rebalance_days": int(round(x[13])),
        "market_filter_lb": int(round(x[14])),
        "leverage": float(x[15]),
    }

    params["top_n"] = max(1, min(params["top_n"], prices_clean.shape[1]))
    params["rebalance_days"] = max(1, params["rebalance_days"])
    params["leverage"] = max(0.0, min(params["leverage"], C3_MAX_LEVERAGE))

    return params


def c3_ga_chromosome_key(chromosome):
    x = c3_ga_repair(chromosome)
    return tuple(np.round(x, 4))


def c3_zscore_cross_sectional(df):
    mean = df.mean(axis=1)
    std = df.std(axis=1).replace(0.0, np.nan)
    return df.sub(mean, axis=0).div(std, axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)


def c3_rsi_matrix(prices, window):
    delta = prices.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)

    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()

    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    rsi = 100.0 - 100.0 / (1.0 + rs)

    return rsi.replace([np.inf, -np.inf], np.nan)


def c3_make_rebalance_mask(index, rebalance_days):
    mask = pd.Series(False, index=index)
    mask.iloc[::rebalance_days] = True
    mask.iloc[0] = True
    return mask


def c3_market_filter_series(prices, lb):
    """
    Filtro de mercado: si la cartera equally-weight está por encima
    de su media móvil, risk-on. Si no, reduce exposición.
    """
    ew_price = prices.mean(axis=1)
    ma = ew_price.rolling(lb).mean()
    risk_on = ew_price > ma
    return risk_on.fillna(False)


def c3_ga_backtest(chromosome):
    p = c3_ga_decode(chromosome)

    prices = prices_clean
    returns = asset_returns

    # Features
    mom_short = prices / prices.shift(p["mom_short_lb"]) - 1.0
    mom_mid = prices / prices.shift(p["mom_mid_lb"]) - 1.0
    mom_long = prices / prices.shift(p["mom_long_lb"]) - 1.0

    vol = returns.rolling(p["vol_lb"]).std()
    low_vol = -vol

    rsi = c3_rsi_matrix(prices, p["rsi_lb"])
    rsi_feature = (rsi - 50.0) / 50.0

    rolling_high = prices.rolling(p["breakout_lb"]).max()
    breakout = prices / rolling_high - 1.0

    score = (
        p["w_mom_short"] * c3_zscore_cross_sectional(mom_short)
        + p["w_mom_mid"] * c3_zscore_cross_sectional(mom_mid)
        + p["w_mom_long"] * c3_zscore_cross_sectional(mom_long)
        + p["w_low_vol"] * c3_zscore_cross_sectional(low_vol)
        + p["w_rsi"] * c3_zscore_cross_sectional(rsi_feature)
        + p["w_breakout"] * c3_zscore_cross_sectional(breakout)
    )

    score = score.where(active_mask, np.nan)

    rebalance_mask = c3_make_rebalance_mask(prices.index, p["rebalance_days"])
    risk_on = c3_market_filter_series(prices, p["market_filter_lb"])

    weights = pd.DataFrame(0.0, index=prices.index, columns=prices.columns)
    current_weights = pd.Series(0.0, index=prices.columns)

    for date in prices.index:
        if rebalance_mask.loc[date]:
            active_scores = score.loc[date].dropna()

            if len(active_scores) > 0:
                n = min(p["top_n"], len(active_scores))
                selected = active_scores.sort_values(ascending=False).head(n).index

                current_weights = pd.Series(0.0, index=prices.columns)

                # Si el filtro de mercado está apagado, reducimos exposición,
                # pero no nos vamos necesariamente a cero.
                exposure = p["leverage"] if risk_on.loc[date] else 0.30 * p["leverage"]

                current_weights.loc[selected] = exposure / n

        weights.loc[date] = current_weights

    # Desfase temporal
    held_weights = weights.shift(1).fillna(0.0)

    gross_returns = (held_weights * returns).sum(axis=1)

    turnover = held_weights.diff().abs().sum(axis=1)
    turnover.iloc[0] = held_weights.iloc[0].abs().sum()

    costs = C3_TRANSACTION_COST * turnover
    net_returns = gross_returns - costs
    net_returns.name = "ga_candidate"

    info = {
        "avg_abs_exposure": float(held_weights.abs().sum(axis=1).mean()),
        "avg_turnover": float(turnover.mean()),
        "total_turnover": float(turnover.sum()),
        "avg_n_positions": float((held_weights.abs() > 0).sum(axis=1).mean()),
        "params": p,
    }

    return net_returns, info


def c3_safe_metric(x, default=0.0):
    try:
        x = float(x)
        if np.isfinite(x):
            return x
        return default
    except Exception:
        return default


def c3_ga_fitness(chromosome):
    """
    Fitness con validación interna:
    - 2021-2022: entrenamiento interno.
    - 2023: validación interna.
    - 2024-2025 no se toca.
    """
    key = c3_ga_chromosome_key(chromosome)

    if key in C3_GA_CACHE:
        return C3_GA_CACHE[key]

    strategy_returns, info = c3_ga_backtest(chromosome)

    # Periodos internos
    r_inner = strategy_returns.loc[inner_train_mask]
    r_val = strategy_returns.loc[validation_mask]
    r_train_total = strategy_returns.loc[train_mask]

    bh_inner = buyhold_returns.loc[inner_train_mask]
    bh_val = buyhold_returns.loc[validation_mask]
    bh_train_total = buyhold_returns.loc[train_mask]

    bm_inner = benchmark_returns.loc[inner_train_mask]
    bm_val = benchmark_returns.loc[validation_mask]
    bm_train_total = benchmark_returns.loc[train_mask]

    # Métricas estrategia
    m_inner = c3_compute_metrics(r_inner, "inner_train")
    m_val = c3_compute_metrics(r_val, "validation")
    m_train_total = c3_compute_metrics(r_train_total, "train_total")

    # Métricas referencias
    bh_m_inner = c3_compute_metrics(bh_inner, "bh_inner")
    bh_m_val = c3_compute_metrics(bh_val, "bh_val")
    bh_m_train = c3_compute_metrics(bh_train_total, "bh_train")

    bm_m_inner = c3_compute_metrics(bm_inner, "bm_inner")
    bm_m_val = c3_compute_metrics(bm_val, "bm_val")
    bm_m_train = c3_compute_metrics(bm_train_total, "bm_train")

    # Estrategia
    cagr_inner = c3_safe_metric(m_inner["CAGR"])
    sharpe_inner = c3_safe_metric(m_inner["Sharpe"])
    sortino_inner = c3_safe_metric(m_inner["Sortino"])
    mdd_inner = c3_safe_metric(m_inner["max_drawdown"], -1.0)

    cagr_val = c3_safe_metric(m_val["CAGR"])
    sharpe_val = c3_safe_metric(m_val["Sharpe"])
    sortino_val = c3_safe_metric(m_val["Sortino"])
    mdd_val = c3_safe_metric(m_val["max_drawdown"], -1.0)

    cagr_train_total = c3_safe_metric(m_train_total["CAGR"])
    sharpe_train_total = c3_safe_metric(m_train_total["Sharpe"])
    sortino_train_total = c3_safe_metric(m_train_total["Sortino"])
    mdd_train_total = c3_safe_metric(m_train_total["max_drawdown"], -1.0)
    calmar_train_total = c3_safe_metric(m_train_total["Calmar"])

    # Referencias
    bh_cagr_inner = c3_safe_metric(bh_m_inner["CAGR"])
    bh_sharpe_inner = c3_safe_metric(bh_m_inner["Sharpe"])
    bh_sortino_inner = c3_safe_metric(bh_m_inner["Sortino"])

    bh_cagr_val = c3_safe_metric(bh_m_val["CAGR"])
    bh_sharpe_val = c3_safe_metric(bh_m_val["Sharpe"])
    bh_sortino_val = c3_safe_metric(bh_m_val["Sortino"])

    bh_cagr_train = c3_safe_metric(bh_m_train["CAGR"])
    bh_sharpe_train = c3_safe_metric(bh_m_train["Sharpe"])
    bh_sortino_train = c3_safe_metric(bh_m_train["Sortino"])

    bm_cagr_inner = c3_safe_metric(bm_m_inner["CAGR"])
    bm_sharpe_inner = c3_safe_metric(bm_m_inner["Sharpe"])
    bm_sortino_inner = c3_safe_metric(bm_m_inner["Sortino"])

    bm_cagr_val = c3_safe_metric(bm_m_val["CAGR"])
    bm_sharpe_val = c3_safe_metric(bm_m_val["Sharpe"])
    bm_sortino_val = c3_safe_metric(bm_m_val["Sortino"])

    bm_cagr_train = c3_safe_metric(bm_m_train["CAGR"])
    bm_sharpe_train = c3_safe_metric(bm_m_train["Sharpe"])
    bm_sortino_train = c3_safe_metric(bm_m_train["Sortino"])

    # Excesos frente a la mejor referencia pasiva
    excess_sharpe_inner = sharpe_inner - max(bh_sharpe_inner, bm_sharpe_inner)
    excess_sortino_inner = sortino_inner - max(bh_sortino_inner, bm_sortino_inner)
    excess_cagr_inner = cagr_inner - max(bh_cagr_inner, bm_cagr_inner)

    excess_sharpe_val = sharpe_val - max(bh_sharpe_val, bm_sharpe_val)
    excess_sortino_val = sortino_val - max(bh_sortino_val, bm_sortino_val)
    excess_cagr_val = cagr_val - max(bh_cagr_val, bm_cagr_val)

    # Penalización por degradación entre inner train y validación
    degradation_sharpe = max(0.0, sharpe_inner - sharpe_val)
    degradation_sortino = max(0.0, sortino_inner - sortino_val)
    degradation_cagr = max(0.0, cagr_inner - cagr_val)

    # Penalizaciones de riesgo
    drawdown_penalty_val = max(0.0, abs(mdd_val) - 0.20)
    drawdown_penalty_train = max(0.0, abs(mdd_train_total) - 0.25)
    turnover_penalty = info["avg_turnover"]

    # Fitness robusto:
    # pesa más validación que entrenamiento interno
    fitness = (
        1.60 * excess_sortino_val
        + 1.30 * excess_sharpe_val
        + 0.90 * excess_cagr_val

        + 0.60 * excess_sortino_inner
        + 0.45 * excess_sharpe_inner
        + 0.30 * excess_cagr_inner

        + 0.40 * sortino_train_total
        + 0.25 * sharpe_train_total
        + 0.20 * cagr_train_total

        - 0.75 * degradation_sortino
        - 0.60 * degradation_sharpe
        - 0.30 * degradation_cagr

        - 0.80 * drawdown_penalty_val
        - 0.40 * drawdown_penalty_train
        - 0.05 * turnover_penalty
    )

    result = {
        "fitness": float(fitness),
        "chromosome": np.array(c3_ga_repair(chromosome), dtype=float),
        "params": c3_ga_decode(chromosome),
        "portfolio_returns": strategy_returns,

        "CAGR_opt": float(cagr_train_total),
        "Sharpe_opt": float(sharpe_train_total),
        "Sortino_opt": float(sortino_train_total),
        "MDD_opt": float(mdd_train_total),
        "Calmar_opt": float(calmar_train_total),

        "CAGR_inner": float(cagr_inner),
        "Sharpe_inner": float(sharpe_inner),
        "Sortino_inner": float(sortino_inner),
        "MDD_inner": float(mdd_inner),

        "CAGR_val": float(cagr_val),
        "Sharpe_val": float(sharpe_val),
        "Sortino_val": float(sortino_val),
        "MDD_val": float(mdd_val),

        "excess_cagr_bh": float(cagr_train_total - bh_cagr_train),
        "excess_sharpe_bh": float(sharpe_train_total - bh_sharpe_train),
        "excess_sortino_bh": float(sortino_train_total - bh_sortino_train),

        "excess_cagr_benchmark": float(cagr_train_total - bm_cagr_train),
        "excess_sharpe_benchmark": float(sharpe_train_total - bm_sharpe_train),
        "excess_sortino_benchmark": float(sortino_train_total - bm_sortino_train),

        "avg_abs_exposure": info["avg_abs_exposure"],
        "avg_turnover": info["avg_turnover"],
        "total_turnover": info["total_turnover"],
        "avg_n_positions": info["avg_n_positions"],
    }

    C3_GA_CACHE[key] = result
    return result


def c3_ga_tournament_select(population, fitness_values, rng):
    idx = rng.choice(len(population), size=C3_GA_TOURNAMENT_SIZE, replace=False)
    best_idx = idx[np.argmax(fitness_values[idx])]
    return population[best_idx].copy()


def c3_ga_uniform_crossover(parent1, parent2, rng):
    if rng.random() > C3_GA_CROSSOVER_PROB:
        return parent1.copy(), parent2.copy()

    mask = rng.random(len(parent1)) < 0.5
    child1 = np.where(mask, parent1, parent2)
    child2 = np.where(mask, parent2, parent1)

    return c3_ga_repair(child1), c3_ga_repair(child2)


def c3_ga_mutate(chromosome, rng):
    x = chromosome.copy()
    ranges = C3_BOUNDS_HIGH - C3_BOUNDS_LOW

    for i in range(len(x)):
        if rng.random() < C3_GA_MUTATION_PROB:
            x[i] += rng.normal(0.0, C3_GA_MUTATION_SCALE * ranges[i])

    return c3_ga_repair(x)


def c3_ga_run():
    population = np.array([
        c3_ga_random_chromosome(rng)
        for _ in range(C3_GA_POPULATION_SIZE)
    ])

    population = np.array([c3_ga_repair(x) for x in population])

    history_rows = []

    for generation in range(C3_GA_GENERATIONS):
        evaluated = [c3_ga_fitness(ind) for ind in population]
        fitness_values = np.array([e["fitness"] for e in evaluated])

        best_idx = int(np.argmax(fitness_values))
        best = evaluated[best_idx]

        history_rows.append({
            "generation": generation,
            "best_fitness": float(np.max(fitness_values)),
            "mean_fitness": float(np.mean(fitness_values)),
            "median_fitness": float(np.median(fitness_values)),

            "best_CAGR_train": best["CAGR_opt"],
            "best_Sharpe_train": best["Sharpe_opt"],
            "best_Sortino_train": best["Sortino_opt"],
            "best_MDD_train": best["MDD_opt"],

            "best_CAGR_inner": best["CAGR_inner"],
            "best_Sharpe_inner": best["Sharpe_inner"],
            "best_Sortino_inner": best["Sortino_inner"],
            "best_MDD_inner": best["MDD_inner"],

            "best_CAGR_val": best["CAGR_val"],
            "best_Sharpe_val": best["Sharpe_val"],
            "best_Sortino_val": best["Sortino_val"],
            "best_MDD_val": best["MDD_val"],

            "best_excess_cagr_bh": best["excess_cagr_bh"],
            "best_excess_sharpe_bh": best["excess_sharpe_bh"],
            "best_excess_sortino_bh": best["excess_sortino_bh"],

            "best_params": json.dumps(best["params"]),
        })

        print(
            f"Gen {generation + 1:02d}/{C3_GA_GENERATIONS} | "
            f"fitness={np.max(fitness_values):.4f} | "
            f"Train Sharpe={best['Sharpe_opt']:.4f} | "
            f"Val Sharpe={best['Sharpe_val']:.4f} | "
            f"Val Sortino={best['Sortino_val']:.4f} | "
            f"Val CAGR={best['CAGR_val']:.4f}"
        )

        elite_indices = np.argsort(fitness_values)[-C3_GA_ELITE_SIZE:]
        elites = population[elite_indices].copy()

        new_population = [e.copy() for e in elites]

        while len(new_population) < C3_GA_POPULATION_SIZE:
            p1 = c3_ga_tournament_select(population, fitness_values, rng)
            p2 = c3_ga_tournament_select(population, fitness_values, rng)

            c1, c2 = c3_ga_uniform_crossover(p1, p2, rng)
            c1 = c3_ga_mutate(c1, rng)
            c2 = c3_ga_mutate(c2, rng)

            new_population.append(c1)

            if len(new_population) < C3_GA_POPULATION_SIZE:
                new_population.append(c2)

        population = np.array(new_population)

    history = pd.DataFrame(history_rows)

    all_results = list(C3_GA_CACHE.values())
    all_results = sorted(all_results, key=lambda x: x["fitness"], reverse=True)

    return history, all_results


C3_GA_CACHE = {}

print("Iniciando GA con validación interna...")
ga_history, ga_results = c3_ga_run()

print("GA terminado.")
print("Individuos únicos evaluados:", len(ga_results))

display(ga_history.tail())


In [ ]:
# ============================================================
# Celda C3.4 - Construir candidatos y seleccionar mejor test
# Narrativa:
# 1) El GA genera candidatos usando train/validación.
# 2) Evaluamos todos los candidatos en test.
# 3) Seleccionamos el mejor observado en test.
# 4) White's Reality Check corregirá por haber probado muchos.


In [ ]:
# ============================================================

top_results = ga_results[:C3_GA_KEEP_TOP]

candidate_returns_dict = {}
candidate_equity_dict = {}
candidate_meta_rows = []

for i, res in enumerate(top_results, start=1):
    name = f"GA_{i:03d}"

    r = res["portfolio_returns"].copy()
    r.name = name

    candidate_returns_dict[name] = r
    candidate_equity_dict[name] = c3_equity_from_returns(r)

    row = {
        "candidate": name,
        "fitness": res.get("fitness", np.nan),
        "params": json.dumps(res.get("params", {})),
        "chromosome": json.dumps(res.get("chromosome", np.array([])).tolist()),

        "CAGR_opt": res.get("CAGR_opt", np.nan),
        "Sharpe_opt": res.get("Sharpe_opt", np.nan),
        "Sortino_opt": res.get("Sortino_opt", np.nan),
        "MDD_opt": res.get("MDD_opt", np.nan),
        "Calmar_opt": res.get("Calmar_opt", np.nan),

        "CAGR_inner": res.get("CAGR_inner", np.nan),
        "Sharpe_inner": res.get("Sharpe_inner", np.nan),
        "Sortino_inner": res.get("Sortino_inner", np.nan),
        "MDD_inner": res.get("MDD_inner", np.nan),

        "CAGR_val": res.get("CAGR_val", np.nan),
        "Sharpe_val": res.get("Sharpe_val", np.nan),
        "Sortino_val": res.get("Sortino_val", np.nan),
        "MDD_val": res.get("MDD_val", np.nan),

        "excess_cagr_bh": res.get("excess_cagr_bh", np.nan),
        "excess_sharpe_bh": res.get("excess_sharpe_bh", np.nan),
        "excess_sortino_bh": res.get("excess_sortino_bh", np.nan),

        "excess_cagr_benchmark": res.get("excess_cagr_benchmark", np.nan),
        "excess_sharpe_benchmark": res.get("excess_sharpe_benchmark", np.nan),
        "excess_sortino_benchmark": res.get("excess_sortino_benchmark", np.nan),

        "avg_abs_exposure": res.get("avg_abs_exposure", np.nan),
        "avg_turnover": res.get("avg_turnover", np.nan),
        "total_turnover": res.get("total_turnover", np.nan),
        "avg_n_positions": res.get("avg_n_positions", np.nan),
    }

    candidate_meta_rows.append(row)

candidate_returns = pd.DataFrame(candidate_returns_dict).sort_index()
candidate_equity = pd.DataFrame(candidate_equity_dict).sort_index()

candidate_meta = pd.DataFrame(candidate_meta_rows)

candidate_metadata = candidate_meta[[
    "candidate",
    "fitness",
    "params",
    "chromosome",
]].copy()

candidate_inner = candidate_returns.loc[inner_train_mask]
candidate_validation = candidate_returns.loc[validation_mask]
candidate_train = candidate_returns.loc[train_mask]
candidate_test = candidate_returns.loc[test_mask]

print("Candidatos conservados:", candidate_returns.shape[1])
print("Observaciones inner train:", candidate_inner.shape[0])
print("Observaciones validación:", candidate_validation.shape[0])
print("Observaciones train total:", candidate_train.shape[0])
print("Observaciones test:", candidate_test.shape[0])

# ------------------------------------------------------------
# Métricas por candidato
# ------------------------------------------------------------

inner_metrics_rows = []
validation_metrics_rows = []
train_metrics_rows = []
test_metrics_rows = []
full_metrics_rows = []

for col in candidate_returns.columns:
    inner_metrics_rows.append(c3_compute_metrics(candidate_inner[col], col))
    validation_metrics_rows.append(c3_compute_metrics(candidate_validation[col], col))
    train_metrics_rows.append(c3_compute_metrics(candidate_train[col], col))
    test_metrics_rows.append(c3_compute_metrics(candidate_test[col], col))
    full_metrics_rows.append(c3_compute_metrics(candidate_returns[col], col))

inner_metrics = pd.DataFrame(inner_metrics_rows)
validation_metrics = pd.DataFrame(validation_metrics_rows)
train_metrics = pd.DataFrame(train_metrics_rows)
test_metrics = pd.DataFrame(test_metrics_rows)
full_metrics = pd.DataFrame(full_metrics_rows)

inner_metrics = inner_metrics.merge(
    candidate_meta,
    left_on="label",
    right_on="candidate",
    how="left"
)

validation_metrics = validation_metrics.merge(
    candidate_meta,
    left_on="label",
    right_on="candidate",
    how="left"
)

train_metrics = train_metrics.merge(
    candidate_meta,
    left_on="label",
    right_on="candidate",
    how="left"
)

test_metrics = test_metrics.merge(
    candidate_meta,
    left_on="label",
    right_on="candidate",
    how="left"
)

full_metrics = full_metrics.merge(
    candidate_meta,
    left_on="label",
    right_on="candidate",
    how="left"
)

# ------------------------------------------------------------
# Selección honesta por validación interna.
# Esta NO será la protagonista, pero la guardamos como referencia.
# ------------------------------------------------------------

candidate_meta_for_selection = candidate_meta.copy()

candidate_meta_for_selection["validation_selection_score"] = (
    candidate_meta_for_selection["fitness"]
    + 0.50 * candidate_meta_for_selection["Sharpe_val"]
    + 0.50 * candidate_meta_for_selection["Sortino_val"]
    - 0.50 * candidate_meta_for_selection["MDD_val"].abs()
    - 0.03 * candidate_meta_for_selection["avg_turnover"]
)

honest_candidate = (
    candidate_meta_for_selection
    .sort_values("validation_selection_score", ascending=False)
    .iloc[0]["candidate"]
)

honest_params = candidate_meta.loc[
    candidate_meta["candidate"] == honest_candidate,
    "params"
].iloc[0]

honest_returns = candidate_returns[honest_candidate].copy()
honest_returns.name = "GA_validation_selected"

honest_equity = c3_equity_from_returns(honest_returns)
honest_equity.name = "GA_validation_selected"

honest_inner_returns = honest_returns.loc[inner_train_mask]
honest_validation_returns = honest_returns.loc[validation_mask]
honest_train_returns = honest_returns.loc[train_mask]
honest_test_returns = honest_returns.loc[test_mask]

print("Estrategia seleccionada por validación interna:", honest_candidate)

# ------------------------------------------------------------
# Ensemble por validación interna.
# También lo guardamos como referencia secundaria.
# ------------------------------------------------------------

if C3_USE_ENSEMBLE:
    ensemble_candidates = (
        candidate_meta_for_selection
        .sort_values("validation_selection_score", ascending=False)
        .head(C3_ENSEMBLE_SIZE)["candidate"]
        .tolist()
    )

    ensemble_returns = candidate_returns[ensemble_candidates].mean(axis=1)
    ensemble_returns.name = "GA_validation_ensemble"

    ensemble_equity = c3_equity_from_returns(ensemble_returns)
    ensemble_equity.name = "GA_validation_ensemble"

    ensemble_inner_returns = ensemble_returns.loc[inner_train_mask]
    ensemble_validation_returns = ensemble_returns.loc[validation_mask]
    ensemble_train_returns = ensemble_returns.loc[train_mask]
    ensemble_test_returns = ensemble_returns.loc[test_mask]

    print("Candidatos del ensemble:")
    print(ensemble_candidates)
else:
    ensemble_candidates = []
    ensemble_returns = None
    ensemble_equity = None
    ensemble_inner_returns = None
    ensemble_validation_returns = None
    ensemble_train_returns = None
    ensemble_test_returns = None

# ------------------------------------------------------------
# Mejor candidato observado en test.
# Esta es la estrategia protagonista del experimento de data snooping.
# ------------------------------------------------------------

test_metrics = test_metrics.copy()

# Score de selección ex post:
# queremos un candidato que sea bueno en test en CAGR, Sharpe y Sortino,
# sin ignorar totalmente el drawdown.
test_metrics["test_selection_score"] = (
    0.40 * test_metrics["Sortino"]
    + 0.30 * test_metrics["Sharpe"]
    + 0.30 * test_metrics["CAGR"]
    - 0.20 * test_metrics["max_drawdown"].abs()
)

best_test_candidate = (
    test_metrics
    .sort_values(
        ["test_selection_score", "Sortino", "Sharpe", "CAGR"],
        ascending=[False, False, False, False]
    )
    .iloc[0]["label"]
)

best_test_params = candidate_meta.loc[
    candidate_meta["candidate"] == best_test_candidate,
    "params"
].iloc[0]

best_test_returns = candidate_returns[best_test_candidate].copy()
best_test_returns.name = "GA_best_test_observed"

best_test_equity = c3_equity_from_returns(best_test_returns)
best_test_equity.name = "GA_best_test_observed"

best_test_inner_returns = best_test_returns.loc[inner_train_mask]
best_test_validation_returns = best_test_returns.loc[validation_mask]
best_test_train_returns = best_test_returns.loc[train_mask]
best_test_test_returns = best_test_returns.loc[test_mask]

print("\nMejor estrategia observada en test, ex post:", best_test_candidate)
print("Parámetros mejor observada en test:")
print(best_test_params)

# ------------------------------------------------------------
# A partir de aquí, la estrategia seleccionada para bootstrap,
# tablas y figuras será la mejor observada en test.
# ------------------------------------------------------------

selected_candidate = best_test_candidate
selected_candidate_label = "GA_best_test_observed"
selected_params = best_test_params

selected_returns = best_test_returns.copy()
selected_returns.name = "GA_best_test_observed"

selected_equity = best_test_equity.copy()
selected_equity.name = "GA_best_test_observed"

selected_train_returns = best_test_train_returns.copy()
selected_test_returns = best_test_test_returns.copy()

print("\nEstrategia usada como selected para C3.5/C3.6:", selected_candidate)
print("Etiqueta visible:", selected_candidate_label)

# ------------------------------------------------------------
# Referencias
# ------------------------------------------------------------

benchmark_inner = benchmark_returns.loc[
    benchmark_returns.index <= pd.Timestamp(C3_INNER_TRAIN_END)
]

benchmark_validation = benchmark_returns.loc[
    (benchmark_returns.index >= pd.Timestamp(C3_VALIDATION_START)) &
    (benchmark_returns.index <= pd.Timestamp(C3_VALIDATION_END))
]

benchmark_train = benchmark_returns.loc[
    benchmark_returns.index <= pd.Timestamp(C3_TRAIN_END)
]

benchmark_test = benchmark_returns.loc[
    benchmark_returns.index >= pd.Timestamp(C3_TEST_START)
]

buyhold_inner = buyhold_returns.loc[
    buyhold_returns.index <= pd.Timestamp(C3_INNER_TRAIN_END)
]

buyhold_validation = buyhold_returns.loc[
    (buyhold_returns.index >= pd.Timestamp(C3_VALIDATION_START)) &
    (buyhold_returns.index <= pd.Timestamp(C3_VALIDATION_END))
]

buyhold_train = buyhold_returns.loc[
    buyhold_returns.index <= pd.Timestamp(C3_TRAIN_END)
]

buyhold_test = buyhold_returns.loc[
    buyhold_returns.index >= pd.Timestamp(C3_TEST_START)
]

# ------------------------------------------------------------
# Métricas finales
# ------------------------------------------------------------

final_metrics_rows = [
    c3_compute_metrics(honest_returns, "GA_validation_selected_full"),
    c3_compute_metrics(honest_inner_returns, "GA_validation_selected_inner_train"),
    c3_compute_metrics(honest_validation_returns, "GA_validation_selected_validation"),
    c3_compute_metrics(honest_train_returns, "GA_validation_selected_train_total"),
    c3_compute_metrics(honest_test_returns, "GA_validation_selected_test"),
]

if C3_USE_ENSEMBLE:
    final_metrics_rows.extend([
        c3_compute_metrics(ensemble_returns, "GA_validation_ensemble_full"),
        c3_compute_metrics(ensemble_inner_returns, "GA_validation_ensemble_inner_train"),
        c3_compute_metrics(ensemble_validation_returns, "GA_validation_ensemble_validation"),
        c3_compute_metrics(ensemble_train_returns, "GA_validation_ensemble_train_total"),
        c3_compute_metrics(ensemble_test_returns, "GA_validation_ensemble_test"),
    ])

final_metrics_rows.extend([
    c3_compute_metrics(best_test_returns, "GA_best_test_observed_full"),
    c3_compute_metrics(best_test_inner_returns, "GA_best_test_observed_inner_train"),
    c3_compute_metrics(best_test_validation_returns, "GA_best_test_observed_validation"),
    c3_compute_metrics(best_test_train_returns, "GA_best_test_observed_train_total"),
    c3_compute_metrics(best_test_test_returns, "GA_best_test_observed_test"),

    c3_compute_metrics(buyhold_returns, "Buy&Hold_full"),
    c3_compute_metrics(buyhold_inner, "Buy&Hold_inner_train"),
    c3_compute_metrics(buyhold_validation, "Buy&Hold_validation"),
    c3_compute_metrics(buyhold_train, "Buy&Hold_train_total"),
    c3_compute_metrics(buyhold_test, "Buy&Hold_test"),

    c3_compute_metrics(benchmark_returns, "Benchmark_full"),
    c3_compute_metrics(benchmark_inner, "Benchmark_inner_train"),
    c3_compute_metrics(benchmark_validation, "Benchmark_validation"),
    c3_compute_metrics(benchmark_train, "Benchmark_train_total"),
    c3_compute_metrics(benchmark_test, "Benchmark_test"),
])

final_metrics = pd.DataFrame(final_metrics_rows)

# ------------------------------------------------------------
# Tabla principal de comparación en test
# ------------------------------------------------------------

main_test_comparison = final_metrics[
    final_metrics["label"].isin([
        "GA_best_test_observed_test",
        "Buy&Hold_test",
        "Benchmark_test",
    ])
].copy()

main_test_comparison["Estrategia"] = main_test_comparison["label"].replace({
    "GA_best_test_observed_test": f"{best_test_candidate} (mejor test)",
    "Buy&Hold_test": "Buy&Hold",
    "Benchmark_test": "Benchmark",
})

main_test_comparison = main_test_comparison[[
    "Estrategia",
    "CAGR",
    "Sharpe",
    "Sortino",
    "max_drawdown",
    "Calmar",
    "cumulative_return",
]]

print("=== Tabla principal: mejor candidato observado en test ===")
display(main_test_comparison)

# ------------------------------------------------------------
# Tabla de proceso de selección
# ------------------------------------------------------------

selection_comparison = pd.DataFrame([
    {
        "selection_type": "validation_selected",
        "candidate": honest_candidate,
        "role": "Selección honesta por validación interna",
        "params": honest_params,
    },
    {
        "selection_type": "validation_ensemble",
        "candidate": "ensemble",
        "role": "Ensemble de candidatos por validación interna",
        "params": json.dumps({"ensemble_candidates": ensemble_candidates}),
    },
    {
        "selection_type": "best_test_observed_ex_post",
        "candidate": best_test_candidate,
        "role": "Mejor candidato observado en test; requiere corrección por data snooping",
        "params": best_test_params,
    },
])

print("=== Proceso de selección ===")
display(selection_comparison)

print("=== Métricas finales completas ===")
display(final_metrics)

print("=== Top 15 en test por test_selection_score ===")
display(
    test_metrics
    .sort_values("test_selection_score", ascending=False)
    [[
        "label",
        "CAGR",
        "Sharpe",
        "Sortino",
        "max_drawdown",
        "Calmar",
        "test_selection_score",
        "fitness",
        "Sharpe_val",
        "Sortino_val",
        "MDD_val",
        "avg_abs_exposure",
        "avg_n_positions",
        "params"
    ]]
    .head(15)
)

print("=== Top 15 en test por Sortino ===")
display(
    test_metrics
    .sort_values("Sortino", ascending=False)
    [[
        "label",
        "CAGR",
        "Sharpe",
        "Sortino",
        "max_drawdown",
        "Calmar",
        "test_selection_score",
        "fitness",
        "Sharpe_val",
        "Sortino_val",
        "MDD_val",
        "avg_abs_exposure",
        "avg_n_positions",
        "params"
    ]]
    .head(15)
)

print("=== Top 15 en validación por Sortino ===")
display(
    validation_metrics
    .sort_values("Sortino", ascending=False)
    [[
        "label",
        "CAGR",
        "Sharpe",
        "Sortino",
        "max_drawdown",
        "Calmar",
        "fitness",
        "Sharpe_val",
        "Sortino_val",
        "MDD_val",
        "avg_abs_exposure",
        "avg_n_positions",
        "params"
    ]]
    .head(15)
)

# Guardar archivos auxiliares
ga_history.to_csv(C3_TABLES_DIR / "C3_ga_history.csv", index=False)
selection_comparison.to_csv(C3_TABLES_DIR / "C3_selection_comparison.csv", index=False)
candidate_meta_for_selection.to_csv(C3_TABLES_DIR / "C3_candidate_selection_scores.csv", index=False)
test_metrics.to_csv(C3_TABLES_DIR / "C3_test_metrics_with_selection_score.csv", index=False)
main_test_comparison.to_csv(C3_TABLES_DIR / "C3_main_test_comparison.csv", index=False)

if C3_USE_ENSEMBLE:
    pd.Series(ensemble_candidates, name="ensemble_candidate").to_csv(
        C3_TABLES_DIR / "C3_ensemble_candidates.csv",
        index=False
    )


In [ ]:
# ============================================================
# Celda C3.5 - Bootstrap y White's Reality Check studentizado


In [ ]:
# ============================================================

print("Ejecutando bootstrap para estrategia seleccionada en evaluación...")
boot_selected, pct_selected = c3_block_bootstrap_metrics(
    selected_test_returns,
    label="GA_best_test_observed_test",
    n_bootstrap=C3_N_BOOTSTRAP,
    block_length=C3_BLOCK_LENGTH,
)

print("Ejecutando bootstrap para Buy&Hold en evaluación...")
boot_buyhold, pct_buyhold = c3_block_bootstrap_metrics(
    buyhold_test,
    label="Buy&Hold_test",
    n_bootstrap=C3_N_BOOTSTRAP,
    block_length=C3_BLOCK_LENGTH,
)

print("Ejecutando bootstrap para benchmark en evaluación...")
boot_benchmark, pct_benchmark = c3_block_bootstrap_metrics(
    benchmark_test,
    label="Benchmark_test",
    n_bootstrap=C3_N_BOOTSTRAP,
    block_length=C3_BLOCK_LENGTH,
)

bootstrap_percentiles = pd.concat(
    [pct_selected, pct_buyhold, pct_benchmark],
    axis=0
)

print("=== Percentiles bootstrap ===")
display(bootstrap_percentiles)


# ------------------------------------------------------------
# White's Reality Check studentizado
# ------------------------------------------------------------

def c3_white_reality_check_studentized(
    candidate_returns: pd.DataFrame,
    reference_returns: pd.Series,
    selected_candidate: str | None = None,
    n_bootstrap: int = C3_WRC_BOOTSTRAP,
    block_length: int = C3_BLOCK_LENGTH,
    random_seed: int = C3_RANDOM_SEED,
):
    """
    White's Reality Check studentizado.

    H0:
    Ninguna estrategia candidata supera a la referencia.

    Estadístico:
    Máximo t-stat de los diferenciales medios frente a la referencia.

    Studentización:
    t_j = mean(d_j) / (std(d_j) / sqrt(n))

    Bootstrap:
    Bootstrap circular por bloques sobre diferenciales centrados.
    En cada muestra bootstrap se calcula el máximo t-stat entre candidatos.
    """

    data = candidate_returns.copy()
    ref = reference_returns.copy()

    common = data.index.intersection(ref.index)

    data = data.loc[common].dropna(axis=1, how="all").fillna(0.0)
    ref = ref.loc[common].fillna(0.0)

    diff = data.sub(ref, axis=0)
    diff = diff.dropna(axis=0, how="all")

    if diff.empty:
        raise ValueError("No hay datos para White's Reality Check studentizado.")

    n = len(diff)
    n_candidates = diff.shape[1]

    mean_diff = diff.mean(axis=0)
    std_diff = diff.std(axis=0, ddof=1).replace(0.0, np.nan)
    se_diff = std_diff / np.sqrt(n)

    t_stats = mean_diff / se_diff
    t_stats = t_stats.replace([np.inf, -np.inf], np.nan)

    # Para elegir el máximo, ignoramos candidatos sin varianza válida.
    t_for_max = t_stats.fillna(-np.inf)

    best_candidate_by_t = t_for_max.idxmax()
    observed_max_t = float(t_for_max.max())

    observed_best_mean_daily = float(mean_diff.loc[best_candidate_by_t])
    observed_best_annualized = observed_best_mean_daily * C3_TRADING_DAYS

    # Diferenciales centrados bajo H0.
    centered = diff - mean_diff
    centered_values = centered.values.astype(float)

    rng = np.random.default_rng(random_seed)

    bootstrap_max_t = []
    bootstrap_max_mean = []

    bootstrap_selected_t = []
    bootstrap_selected_mean = []

    selected_idx = None

    if selected_candidate is not None and selected_candidate in diff.columns:
        selected_idx = list(diff.columns).index(selected_candidate)
        selected_t = float(t_stats.loc[selected_candidate])
        selected_mean_daily = float(mean_diff.loc[selected_candidate])
        selected_annualized = selected_mean_daily * C3_TRADING_DAYS
    else:
        selected_t = np.nan
        selected_mean_daily = np.nan
        selected_annualized = np.nan

    for b in range(n_bootstrap):
        idx = c3_circular_block_indices(n, block_length, rng)
        sample = centered_values[idx, :]

        sample_means = np.nanmean(sample, axis=0)
        sample_stds = np.nanstd(sample, axis=0, ddof=1)
        sample_ses = sample_stds / np.sqrt(n)

        with np.errstate(divide="ignore", invalid="ignore"):
            sample_t = sample_means / sample_ses

        sample_t = np.where(np.isfinite(sample_t), sample_t, -np.inf)

        bootstrap_max_t.append(float(np.max(sample_t)))
        bootstrap_max_mean.append(float(np.max(sample_means)))

        if selected_idx is not None:
            bootstrap_selected_t.append(float(sample_t[selected_idx]))
            bootstrap_selected_mean.append(float(sample_means[selected_idx]))

    bootstrap_max_t = np.array(bootstrap_max_t, dtype=float)
    bootstrap_max_mean = np.array(bootstrap_max_mean, dtype=float)

    p_value_corrected_family = float(np.mean(bootstrap_max_t >= observed_max_t))

    if selected_idx is not None and np.isfinite(selected_t):
        bootstrap_selected_t = np.array(bootstrap_selected_t, dtype=float)
        bootstrap_selected_mean = np.array(bootstrap_selected_mean, dtype=float)

        p_value_individual_selected = float(np.mean(bootstrap_selected_t >= selected_t))
        p_value_corrected_selected = float(np.mean(bootstrap_max_t >= selected_t))
    else:
        p_value_individual_selected = np.nan
        p_value_corrected_selected = np.nan

    summary = pd.DataFrame([{
        "best_candidate_by_t": best_candidate_by_t,
        "observed_max_t": observed_max_t,
        "observed_best_mean_daily_excess_return": observed_best_mean_daily,
        "observed_best_annualized_excess_return": observed_best_annualized,
        "p_value_corrected_family": p_value_corrected_family,
        "n_observations": int(n),
        "n_candidates": int(n_candidates),
        "n_bootstrap": int(n_bootstrap),
        "block_length": int(block_length),
    }])

    candidate_table = pd.DataFrame({
        "candidate": mean_diff.index,
        "mean_daily_excess_return": mean_diff.values,
        "annualized_excess_return": mean_diff.values * C3_TRADING_DAYS,
        "std_daily_excess_return": std_diff.values,
        "standard_error": se_diff.values,
        "t_stat": t_stats.values,
    }).sort_values("t_stat", ascending=False)

    boot_table = pd.DataFrame({
        "bootstrap_max_t": bootstrap_max_t,
        "bootstrap_max_mean": bootstrap_max_mean,
    })

    if selected_idx is not None:
        boot_table["bootstrap_selected_t"] = bootstrap_selected_t
        boot_table["bootstrap_selected_mean"] = bootstrap_selected_mean

    selected_summary = pd.DataFrame([{
        "selected_candidate": selected_candidate,
        "best_candidate_by_t": best_candidate_by_t,
        "observed_mean_daily_excess_return": selected_mean_daily,
        "observed_annualized_excess_return": selected_annualized,
        "observed_t_stat": selected_t,
        "p_value_individual": p_value_individual_selected,
        "p_value_corrected": p_value_corrected_selected,
        "n_observations": int(n),
        "n_candidates": int(n_candidates),
        "n_bootstrap": int(n_bootstrap),
        "block_length": int(block_length),
    }])

    return summary, candidate_table, boot_table, selected_summary


# ------------------------------------------------------------
# Aplicar WRC studentizado en periodo test
# ------------------------------------------------------------

candidate_test_aligned = candidate_returns.loc[test_mask].copy()

print("Ejecutando White's Reality Check studentizado frente a benchmark...")
(
    wrc_benchmark_student_summary,
    wrc_benchmark_student_candidates,
    wrc_benchmark_student_boot,
    selected_benchmark_student_summary,
) = c3_white_reality_check_studentized(
    candidate_returns=candidate_test_aligned,
    reference_returns=benchmark_test,
    selected_candidate=selected_candidate,
    n_bootstrap=C3_WRC_BOOTSTRAP,
    block_length=C3_BLOCK_LENGTH,
)

print("Ejecutando White's Reality Check studentizado frente a Buy&Hold...")
(
    wrc_buyhold_student_summary,
    wrc_buyhold_student_candidates,
    wrc_buyhold_student_boot,
    selected_buyhold_student_summary,
) = c3_white_reality_check_studentized(
    candidate_returns=candidate_test_aligned,
    reference_returns=buyhold_test,
    selected_candidate=selected_candidate,
    n_bootstrap=C3_WRC_BOOTSTRAP,
    block_length=C3_BLOCK_LENGTH,
)

wrc_benchmark_student_summary.insert(0, "reference", "Benchmark")
wrc_buyhold_student_summary.insert(0, "reference", "Buy&Hold")

selected_benchmark_student_summary.insert(0, "reference", "Benchmark")
selected_buyhold_student_summary.insert(0, "reference", "Buy&Hold")

wrc_studentized_summary = pd.concat(
    [wrc_benchmark_student_summary, wrc_buyhold_student_summary],
    ignore_index=True
)

selected_expost_wrc_studentized_summary = pd.concat(
    [selected_benchmark_student_summary, selected_buyhold_student_summary],
    ignore_index=True
)

# Para compatibilidad con C3.6 y C3.7 antiguas:
wrc_summary = wrc_studentized_summary.copy()
selected_expost_wrc_summary = selected_expost_wrc_studentized_summary.copy()

wrc_benchmark_summary = wrc_benchmark_student_summary.copy()
wrc_buyhold_summary = wrc_buyhold_student_summary.copy()

wrc_benchmark_candidates = wrc_benchmark_student_candidates.copy()
wrc_buyhold_candidates = wrc_buyhold_student_candidates.copy()

wrc_benchmark_boot = wrc_benchmark_student_boot.copy()
wrc_buyhold_boot = wrc_buyhold_student_boot.copy()

print("=== White's Reality Check studentizado: familia completa ===")
display(wrc_studentized_summary)

print("=== White's Reality Check studentizado: candidato seleccionado ===")
display(selected_expost_wrc_studentized_summary)

print("=== Top candidatos por t-stat frente a benchmark ===")
display(wrc_benchmark_student_candidates.head(10))

print("=== Top candidatos por t-stat frente a Buy&Hold ===")
display(wrc_buyhold_student_candidates.head(10))


In [ ]:
# ============================================================
# Celda C3.6 - Guardar tablas, metadata y figuras
# Narrativa centrada en mejor candidato observado en test


In [ ]:
# ============================================================

# ------------------------------------------------------------
# Guardar series principales
# ------------------------------------------------------------

candidate_returns.to_csv(C3_TABLES_DIR / "C3_candidate_returns.csv")
candidate_equity.to_csv(C3_TABLES_DIR / "C3_candidate_equity.csv")

selected_returns.to_csv(C3_TABLES_DIR / "C3_selected_best_test_returns.csv")
selected_equity.to_csv(C3_TABLES_DIR / "C3_selected_best_test_equity.csv")

if "honest_returns" in globals():
    honest_returns.to_csv(C3_TABLES_DIR / "C3_validation_selected_returns.csv")

if "best_test_returns" in globals():
    best_test_returns.to_csv(C3_TABLES_DIR / "C3_best_test_observed_returns.csv")

if "ensemble_returns" in globals() and ensemble_returns is not None:
    ensemble_returns.to_csv(C3_TABLES_DIR / "C3_validation_ensemble_returns.csv")

# ------------------------------------------------------------
# Guardar tablas base
# ------------------------------------------------------------

candidate_metadata.to_csv(C3_TABLES_DIR / "C3_candidate_metadata.csv", index=False)
candidate_meta.to_csv(C3_TABLES_DIR / "C3_candidate_backtest_metadata.csv", index=False)

if "candidate_meta_for_selection" in globals():
    candidate_meta_for_selection.to_csv(
        C3_TABLES_DIR / "C3_candidate_selection_scores.csv",
        index=False
    )

if "selection_comparison" in globals():
    selection_comparison.to_csv(
        C3_TABLES_DIR / "C3_selection_comparison.csv",
        index=False
    )

if "inner_metrics" in globals():
    inner_metrics.to_csv(C3_TABLES_DIR / "C3_inner_train_metrics.csv", index=False)

if "validation_metrics" in globals():
    validation_metrics.to_csv(C3_TABLES_DIR / "C3_validation_metrics.csv", index=False)

train_metrics.to_csv(C3_TABLES_DIR / "C3_train_metrics.csv", index=False)
test_metrics.to_csv(C3_TABLES_DIR / "C3_test_metrics.csv", index=False)
full_metrics.to_csv(C3_TABLES_DIR / "C3_full_metrics.csv", index=False)
final_metrics.to_csv(C3_TABLES_DIR / "C3_final_metrics.csv", index=False)
main_test_comparison.to_csv(C3_TABLES_DIR / "C3_main_test_comparison.csv", index=False)

if "ga_history" in globals():
    ga_history.to_csv(C3_TABLES_DIR / "C3_ga_history.csv", index=False)

if "ensemble_candidates" in globals():
    pd.Series(ensemble_candidates, name="ensemble_candidate").to_csv(
        C3_TABLES_DIR / "C3_ensemble_candidates.csv",
        index=False
    )

# ------------------------------------------------------------
# Guardar resultados de bootstrap y White's Reality Check
# ------------------------------------------------------------

boot_selected.to_csv(C3_TABLES_DIR / "C3_bootstrap_selected_best_test.csv", index=False)
boot_buyhold.to_csv(C3_TABLES_DIR / "C3_bootstrap_buyhold_test.csv", index=False)
boot_benchmark.to_csv(C3_TABLES_DIR / "C3_bootstrap_benchmark_test.csv", index=False)
bootstrap_percentiles.to_csv(C3_TABLES_DIR / "C3_bootstrap_percentiles.csv")

wrc_summary.to_csv(C3_TABLES_DIR / "C3_white_reality_check_summary.csv", index=False)
wrc_benchmark_candidates.to_csv(C3_TABLES_DIR / "C3_wrc_candidates_vs_benchmark.csv", index=False)
wrc_buyhold_candidates.to_csv(C3_TABLES_DIR / "C3_wrc_candidates_vs_buyhold.csv", index=False)
wrc_benchmark_boot.to_csv(C3_TABLES_DIR / "C3_wrc_bootstrap_vs_benchmark.csv", index=False)
wrc_buyhold_boot.to_csv(C3_TABLES_DIR / "C3_wrc_bootstrap_vs_buyhold.csv", index=False)

# ------------------------------------------------------------
# Tabla específica:
# p-value corregido por familia para el candidato seleccionado ex post
# ------------------------------------------------------------

def c3_family_adjusted_pvalue_for_selected(selected_test, reference_test, boot_df):
    common_index = selected_test.index.intersection(reference_test.index)
    excess = selected_test.loc[common_index] - reference_test.loc[common_index]

    observed_daily = float(excess.mean())
    observed_annualized = observed_daily * 252.0

    p_value = float(
        (boot_df["bootstrap_max_mean"] >= observed_daily).mean()
    )

    return observed_daily, observed_annualized, p_value


selected_vs_benchmark_daily, selected_vs_benchmark_ann, selected_vs_benchmark_p = (
    c3_family_adjusted_pvalue_for_selected(
        selected_test_returns,
        benchmark_test,
        wrc_benchmark_boot
    )
)

selected_vs_buyhold_daily, selected_vs_buyhold_ann, selected_vs_buyhold_p = (
    c3_family_adjusted_pvalue_for_selected(
        selected_test_returns,
        buyhold_test,
        wrc_buyhold_boot
    )
)

selected_expost_wrc_summary = pd.DataFrame([
    {
        "selected_candidate": selected_candidate,
        "visible_label": "GA_best_test_observed",
        "reference": "Benchmark",
        "observed_mean_daily_excess_return": selected_vs_benchmark_daily,
        "observed_annualized_excess_return": selected_vs_benchmark_ann,
        "family_adjusted_p_value": selected_vs_benchmark_p,
    },
    {
        "selected_candidate": selected_candidate,
        "visible_label": "GA_best_test_observed",
        "reference": "Buy&Hold",
        "observed_mean_daily_excess_return": selected_vs_buyhold_daily,
        "observed_annualized_excess_return": selected_vs_buyhold_ann,
        "family_adjusted_p_value": selected_vs_buyhold_p,
    },
])

selected_expost_wrc_summary.to_csv(
    C3_TABLES_DIR / "C3_selected_expost_wrc_summary.csv",
    index=False
)

print("=== White corregido para el candidato seleccionado ex post ===")
display(selected_expost_wrc_summary)

# ------------------------------------------------------------
# Tablas LaTeX limpias para el TFG
# ------------------------------------------------------------

def c3_fmt_pct(x):
    if pd.isna(x):
        return ""
    return f"{100.0 * float(x):.2f}\\%".replace(".", ",")


def c3_fmt_num(x):
    if pd.isna(x):
        return ""
    return f"{float(x):.3f}".replace(".", ",")


def c3_fmt_p(x):
    if pd.isna(x):
        return ""
    return f"{float(x):.3f}".replace(".", ",")


# Tabla principal de test
latex_main_test = main_test_comparison.copy()

latex_main_test["CAGR"] = latex_main_test["CAGR"].map(c3_fmt_pct)
latex_main_test["Sharpe"] = latex_main_test["Sharpe"].map(c3_fmt_num)
latex_main_test["Sortino"] = latex_main_test["Sortino"].map(c3_fmt_num)
latex_main_test["max_drawdown"] = latex_main_test["max_drawdown"].map(c3_fmt_pct)
latex_main_test["Calmar"] = latex_main_test["Calmar"].map(c3_fmt_num)
latex_main_test["cumulative_return"] = latex_main_test["cumulative_return"].map(c3_fmt_pct)

latex_main_test = latex_main_test.rename(columns={
    "max_drawdown": "MDD",
    "cumulative_return": "Ret. acumulada",
})

latex_main_test.to_latex(
    C3_TABLES_DIR / "C3_table_main_test_comparison.tex",
    index=False,
    escape=False,
    caption="Comparación fuera de muestra del mejor candidato observado en test frente a las referencias pasivas.",
    label="tab:c3_main_test_comparison"
)

# Tabla White's Reality Check para seleccionado ex post
latex_wrc_selected = selected_expost_wrc_summary.copy()
latex_wrc_selected["observed_annualized_excess_return"] = (
    latex_wrc_selected["observed_annualized_excess_return"].map(c3_fmt_pct)
)
latex_wrc_selected["family_adjusted_p_value"] = (
    latex_wrc_selected["family_adjusted_p_value"].map(c3_fmt_p)
)

latex_wrc_selected = latex_wrc_selected[[
    "selected_candidate",
    "reference",
    "observed_annualized_excess_return",
    "family_adjusted_p_value",
]].rename(columns={
    "selected_candidate": "Candidato",
    "reference": "Referencia",
    "observed_annualized_excess_return": "Exceso anualizado",
    "family_adjusted_p_value": "p-value corregido",
})

latex_wrc_selected.to_latex(
    C3_TABLES_DIR / "C3_table_selected_expost_wrc.tex",
    index=False,
    escape=False,
    caption="White's Reality Check aplicado al mejor candidato observado en test.",
    label="tab:c3_selected_expost_wrc"
)

# Tabla proceso de selección
latex_selection = selection_comparison[[
    "selection_type",
    "candidate",
    "role",
]].copy()

latex_selection = latex_selection.rename(columns={
    "selection_type": "Tipo",
    "candidate": "Candidato",
    "role": "Interpretación",
})

latex_selection.to_latex(
    C3_TABLES_DIR / "C3_table_selection_process.tex",
    index=False,
    escape=False,
    caption="Proceso de selección de estrategias en el Módulo C3.",
    label="tab:c3_selection_process"
)

# ------------------------------------------------------------
# Metadata
# ------------------------------------------------------------

metadata = {
    "module": "C3_genetic_algorithm_data_snooping",
    "description": (
        "Algoritmo genético cross-sectional. Los candidatos se generan con train/validación. "
        "Después se evalúan en test y se identifica el mejor candidato observado ex post. "
        "White's Reality Check corrige por la búsqueda entre múltiples estrategias."
    ),
    "start_date": CONFIG.start_date,
    "end_date": CONFIG.end_date,
    "inner_train_end": globals().get("C3_INNER_TRAIN_END", None),
    "validation_start": globals().get("C3_VALIDATION_START", None),
    "validation_end": globals().get("C3_VALIDATION_END", None),
    "train_end": C3_TRAIN_END,
    "test_start": C3_TEST_START,
    "universe": "S&P 100 historical reconstruction from C0",
    "benchmark_ticker": CONFIG.benchmark_ticker,
    "transaction_cost": C3_TRANSACTION_COST,
    "risk_free_rate_annual": C3_RISK_FREE_RATE_ANNUAL,
    "n_assets": int(prices_clean.shape[1]),
    "n_observations": int(prices_clean.shape[0]),
    "n_candidates": int(candidate_returns.shape[1]),
    "ga_population_size": globals().get("C3_GA_POPULATION_SIZE", None),
    "ga_generations": globals().get("C3_GA_GENERATIONS", None),
    "ga_keep_top": globals().get("C3_GA_KEEP_TOP", None),
    "selected_candidate": selected_candidate,
    "selected_visible_label": "GA_best_test_observed",
    "selected_params": selected_params,
    "bootstrap_n": C3_N_BOOTSTRAP,
    "white_reality_check_n": C3_WRC_BOOTSTRAP,
    "block_length": C3_BLOCK_LENGTH,
    "selected_expost_wrc_summary": selected_expost_wrc_summary.to_dict(orient="records"),
    "output_dir": str(C3_OUTPUT_DIR),
}

with open(C3_OUTPUT_DIR / "C3_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

# ------------------------------------------------------------
# Figuras principales
# ------------------------------------------------------------

# Equity full con inicio de test
selected_equity_full = c3_equity_from_returns(selected_returns)
buyhold_equity_full = c3_equity_from_returns(buyhold_returns)
benchmark_equity_full = c3_equity_from_returns(benchmark_returns)

plt.figure(figsize=(12, 6))
selected_equity_full.plot(label=f"{selected_candidate} mejor test")
buyhold_equity_full.plot(label="Buy&Hold")
benchmark_equity_full.plot(label="Benchmark")
plt.axvline(pd.Timestamp(C3_TEST_START), linestyle="--", linewidth=1, label="Inicio test")
plt.title("Módulo C3: curvas de capital completas")
plt.xlabel("Fecha")
plt.ylabel("Capital normalizado")
plt.legend()
plt.tight_layout()
plt.savefig(C3_FIGURES_DIR / "C3_equity_curves_full.png", dpi=160)
plt.close()

# Equity solo test, renormalizado a 1
test_equity_df = pd.DataFrame({
    f"{selected_candidate} mejor test": c3_equity_from_returns(selected_test_returns),
    "Buy&Hold": c3_equity_from_returns(buyhold_test),
    "Benchmark": c3_equity_from_returns(benchmark_test),
}).dropna()

test_equity_df = test_equity_df / test_equity_df.iloc[0]

plt.figure(figsize=(12, 6))
test_equity_df.plot(ax=plt.gca())
plt.title("Módulo C3: curvas de capital en el periodo test")
plt.xlabel("Fecha")
plt.ylabel("Capital normalizado")
plt.legend()
plt.tight_layout()
plt.savefig(C3_FIGURES_DIR / "C3_equity_curves_test_focus.png", dpi=160)
plt.close()

# Drawdown solo test
plt.figure(figsize=(12, 6))
for col in test_equity_df.columns:
    c3_drawdown_series(test_equity_df[col]).plot(label=col)
plt.title("Módulo C3: drawdowns en el periodo test")
plt.xlabel("Fecha")
plt.ylabel("Drawdown")
plt.legend()
plt.tight_layout()
plt.savefig(C3_FIGURES_DIR / "C3_drawdowns_test_focus.png", dpi=160)
plt.close()

# Drawdown full
plt.figure(figsize=(12, 6))
c3_drawdown_series(selected_equity_full).plot(label=f"{selected_candidate} mejor test")
c3_drawdown_series(buyhold_equity_full).plot(label="Buy&Hold")
c3_drawdown_series(benchmark_equity_full).plot(label="Benchmark")
plt.axvline(pd.Timestamp(C3_TEST_START), linestyle="--", linewidth=1, label="Inicio test")
plt.title("Módulo C3: drawdowns completos")
plt.xlabel("Fecha")
plt.ylabel("Drawdown")
plt.legend()
plt.tight_layout()
plt.savefig(C3_FIGURES_DIR / "C3_drawdowns_full.png", dpi=160)
plt.close()

# Scatter de candidatos en test: CAGR vs Sharpe
plt.figure(figsize=(10, 6))
plt.scatter(test_metrics["Sharpe"], test_metrics["CAGR"], alpha=0.65)
selected_row = test_metrics.loc[test_metrics["label"] == selected_candidate].iloc[0]
plt.scatter(
    [selected_row["Sharpe"]],
    [selected_row["CAGR"]],
    marker="*",
    s=220,
    label=f"{selected_candidate} seleccionado"
)
plt.axhline(float(final_metrics.loc[final_metrics["label"] == "Buy&Hold_test", "CAGR"].iloc[0]), linestyle="--", linewidth=1, label="CAGR Buy&Hold")
plt.axhline(float(final_metrics.loc[final_metrics["label"] == "Benchmark_test", "CAGR"].iloc[0]), linestyle=":", linewidth=1, label="CAGR Benchmark")
plt.title("Módulo C3: candidatos en test")
plt.xlabel("Sharpe en test")
plt.ylabel("CAGR en test")
plt.legend()
plt.tight_layout()
plt.savefig(C3_FIGURES_DIR / "C3_candidates_test_cagr_sharpe.png", dpi=160)
plt.close()

# Validación vs test Sortino
if "validation_metrics" in globals():
    merged_val_test = validation_metrics[["label", "Sortino"]].merge(
        test_metrics[["label", "Sortino"]],
        on="label",
        suffixes=("_validation", "_test")
    )

    plt.figure(figsize=(10, 6))
    plt.scatter(
        merged_val_test["Sortino_validation"],
        merged_val_test["Sortino_test"],
        alpha=0.65
    )

    selected_val_test = merged_val_test.loc[
        merged_val_test["label"] == selected_candidate
    ].iloc[0]

    plt.scatter(
        [selected_val_test["Sortino_validation"]],
        [selected_val_test["Sortino_test"]],
        marker="*",
        s=220,
        label=f"{selected_candidate} seleccionado"
    )

    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.title("Módulo C3: Sortino validación interna vs test")
    plt.xlabel("Sortino en validación interna")
    plt.ylabel("Sortino en test")
    plt.legend()
    plt.tight_layout()
    plt.savefig(C3_FIGURES_DIR / "C3_validation_vs_test_sortino.png", dpi=160)
    plt.close()

# White's Reality Check frente a benchmark
plt.figure(figsize=(10, 6))
wrc_benchmark_boot["bootstrap_max_mean"].plot(kind="hist", bins=40)
plt.axvline(
    selected_vs_benchmark_daily,
    linestyle="--",
    linewidth=2,
    label="Exceso observado del candidato seleccionado"
)
plt.title("White's Reality Check frente a benchmark")
plt.xlabel("Máximo retorno diferencial medio bootstrap")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.savefig(C3_FIGURES_DIR / "C3_wrc_vs_benchmark.png", dpi=160)
plt.close()

# White's Reality Check frente a Buy&Hold
plt.figure(figsize=(10, 6))
wrc_buyhold_boot["bootstrap_max_mean"].plot(kind="hist", bins=40)
plt.axvline(
    selected_vs_buyhold_daily,
    linestyle="--",
    linewidth=2,
    label="Exceso observado del candidato seleccionado"
)
plt.title("White's Reality Check frente a Buy&Hold")
plt.xlabel("Máximo retorno diferencial medio bootstrap")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.savefig(C3_FIGURES_DIR / "C3_wrc_vs_buyhold.png", dpi=160)
plt.close()

print("Resultados guardados en:", C3_OUTPUT_DIR)
print("Tablas:", C3_TABLES_DIR)
print("Figuras:", C3_FIGURES_DIR)
print("Candidato seleccionado ex post:", selected_candidate)
print("Etiqueta visible:", "GA_best_test_observed")


In [ ]:
# ============================================================
# Celda C3.6b - Guardar White's Reality Check studentizado


In [ ]:
# ============================================================

# ------------------------------------------------------------
# Comprobaciones
# ------------------------------------------------------------

required_studentized_objects = [
    "wrc_studentized_summary",
    "selected_expost_wrc_studentized_summary",
    "wrc_benchmark_student_candidates",
    "wrc_buyhold_student_candidates",
    "wrc_benchmark_student_boot",
    "wrc_buyhold_student_boot",
]

missing = [x for x in required_studentized_objects if x not in globals()]

if len(missing) > 0:
    raise RuntimeError(
        "Faltan objetos del White studentizado. Ejecuta primero la C3.5 nueva.\n"
        f"Faltan: {missing}"
    )

# ------------------------------------------------------------
# Guardar CSV studentizados
# ------------------------------------------------------------

wrc_studentized_summary.to_csv(
    C3_TABLES_DIR / "C3_wrc_studentized_family_summary.csv",
    index=False
)

selected_expost_wrc_studentized_summary.to_csv(
    C3_TABLES_DIR / "C3_selected_expost_wrc_studentized_summary.csv",
    index=False
)

# Sobrescribimos también el nombre antiguo para que el notebook use la versión buena.
selected_expost_wrc_studentized_summary.to_csv(
    C3_TABLES_DIR / "C3_selected_expost_wrc_summary.csv",
    index=False
)

wrc_benchmark_student_candidates.to_csv(
    C3_TABLES_DIR / "C3_wrc_studentized_candidates_vs_benchmark.csv",
    index=False
)

wrc_buyhold_student_candidates.to_csv(
    C3_TABLES_DIR / "C3_wrc_studentized_candidates_vs_buyhold.csv",
    index=False
)

wrc_benchmark_student_boot.to_csv(
    C3_TABLES_DIR / "C3_wrc_studentized_bootstrap_vs_benchmark.csv",
    index=False
)

wrc_buyhold_student_boot.to_csv(
    C3_TABLES_DIR / "C3_wrc_studentized_bootstrap_vs_buyhold.csv",
    index=False
)

# ------------------------------------------------------------
# Formateadores LaTeX
# ------------------------------------------------------------

def c3_fmt_pct_latex(x):
    if pd.isna(x):
        return ""
    return f"{100.0 * float(x):.2f}\\%".replace(".", ",")


def c3_fmt_num_latex(x):
    if pd.isna(x):
        return ""
    return f"{float(x):.3f}".replace(".", ",")


# ------------------------------------------------------------
# Tabla LaTeX principal para el TFG
# ------------------------------------------------------------

latex_wrc_studentized = selected_expost_wrc_studentized_summary.copy()

latex_wrc_studentized["Contraste"] = (
    latex_wrc_studentized["selected_candidate"].astype(str)
    + " vs "
    + latex_wrc_studentized["reference"].astype(str)
)

latex_wrc_studentized["Periodo"] = "Test 2024--2025"

latex_wrc_studentized["Exceso anualizado"] = (
    latex_wrc_studentized["observed_annualized_excess_return"].map(c3_fmt_pct_latex)
)

latex_wrc_studentized["t observado"] = (
    latex_wrc_studentized["observed_t_stat"].map(c3_fmt_num_latex)
)

latex_wrc_studentized["p-value individual"] = (
    latex_wrc_studentized["p_value_individual"].map(c3_fmt_num_latex)
)

latex_wrc_studentized["p-value corregido"] = (
    latex_wrc_studentized["p_value_corrected"].map(c3_fmt_num_latex)
)

latex_wrc_studentized["Mejor estrategia por t"] = (
    latex_wrc_studentized["best_candidate_by_t"].astype(str)
)

latex_wrc_studentized = latex_wrc_studentized[[
    "Contraste",
    "Periodo",
    "Exceso anualizado",
    "t observado",
    "p-value individual",
    "p-value corregido",
    "Mejor estrategia por t",
]]

latex_wrc_studentized.to_latex(
    C3_TABLES_DIR / "C3_table_selected_expost_wrc.tex",
    index=False,
    escape=False,
    caption="White's Reality Check studentizado aplicado al mejor candidato observado en test.",
    label="tab:c3_selected_expost_wrc"
)

latex_wrc_studentized.to_latex(
    C3_TABLES_DIR / "C3_table_selected_expost_wrc_studentized.tex",
    index=False,
    escape=False,
    caption="White's Reality Check studentizado aplicado al mejor candidato observado en test.",
    label="tab:c3_selected_expost_wrc_studentized"
)

# ------------------------------------------------------------
# Figuras studentizadas
# ------------------------------------------------------------

selected_vs_benchmark_t = float(
    selected_expost_wrc_studentized_summary.loc[
        selected_expost_wrc_studentized_summary["reference"] == "Benchmark",
        "observed_t_stat"
    ].iloc[0]
)

selected_vs_buyhold_t = float(
    selected_expost_wrc_studentized_summary.loc[
        selected_expost_wrc_studentized_summary["reference"] == "Buy&Hold",
        "observed_t_stat"
    ].iloc[0]
)

plt.figure(figsize=(10, 6))
wrc_benchmark_student_boot["bootstrap_max_t"].plot(kind="hist", bins=40)
plt.axvline(
    selected_vs_benchmark_t,
    linestyle="--",
    linewidth=2,
    label="t observado del candidato seleccionado"
)
plt.title("White's Reality Check studentizado frente a benchmark")
plt.xlabel("Máximo t-stat bootstrap")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.savefig(C3_FIGURES_DIR / "C3_wrc_studentized_vs_benchmark.png", dpi=160)
plt.close()

plt.figure(figsize=(10, 6))
wrc_buyhold_student_boot["bootstrap_max_t"].plot(kind="hist", bins=40)
plt.axvline(
    selected_vs_buyhold_t,
    linestyle="--",
    linewidth=2,
    label="t observado del candidato seleccionado"
)
plt.title("White's Reality Check studentizado frente a Buy&Hold")
plt.xlabel("Máximo t-stat bootstrap")
plt.ylabel("Frecuencia")
plt.legend()
plt.tight_layout()
plt.savefig(C3_FIGURES_DIR / "C3_wrc_studentized_vs_buyhold.png", dpi=160)
plt.close()

# ------------------------------------------------------------
# Actualizar metadata
# ------------------------------------------------------------

metadata_path = C3_OUTPUT_DIR / "C3_metadata.json"

if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
else:
    metadata = {}

metadata["white_reality_check_type"] = "studentized"
metadata["selected_expost_wrc_studentized_summary"] = (
    selected_expost_wrc_studentized_summary.to_dict(orient="records")
)
metadata["wrc_studentized_family_summary"] = (
    wrc_studentized_summary.to_dict(orient="records")
)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print("=== White's Reality Check studentizado guardado ===")
print("Tabla CSV:", C3_TABLES_DIR / "C3_selected_expost_wrc_studentized_summary.csv")
print("Tabla LaTeX:", C3_TABLES_DIR / "C3_table_selected_expost_wrc_studentized.tex")
print("Figura benchmark:", C3_FIGURES_DIR / "C3_wrc_studentized_vs_benchmark.png")
print("Figura Buy&Hold:", C3_FIGURES_DIR / "C3_wrc_studentized_vs_buyhold.png")

display(latex_wrc_studentized)


In [ ]:
# ============================================================
# Celda C3.7 - Ver resultados finales del Módulo C3


In [ ]:
# ============================================================

from IPython.display import display, Image

print("============================================================")
print("MÓDULO C3 - RESUMEN FINAL")
print("============================================================")
print("Candidato seleccionado ex post:", selected_candidate)
print("Etiqueta visible:", "GA_best_test_observed")
print("Carpeta de resultados:", C3_OUTPUT_DIR)

print("\n=== Tabla principal: test ===")
display(main_test_comparison)

print("\n=== White's Reality Check corregido para el candidato ex post ===")
display(selected_expost_wrc_summary)

print("\n=== Proceso de selección ===")
display(selection_comparison)

print("\n=== Top 10 candidatos en test por score ===")
display(
    test_metrics
    .sort_values("test_selection_score", ascending=False)
    [[
        "label",
        "CAGR",
        "Sharpe",
        "Sortino",
        "max_drawdown",
        "Calmar",
        "test_selection_score",
        "params"
    ]]
    .head(10)
)

print("\n=== Figuras principales ===")

figures_to_show = [
    "C3_equity_curves_test_focus.png",
    "C3_drawdowns_test_focus.png",
    "C3_candidates_test_cagr_sharpe.png",
    "C3_validation_vs_test_sortino.png",
    "C3_wrc_vs_buyhold.png",
    "C3_wrc_vs_benchmark.png",
]

for fig in figures_to_show:
    path = C3_FIGURES_DIR / fig
    if path.exists():
        print("\n", fig)
        display(Image(filename=str(path)))
    else:
        print("No encontrada:", fig)

print("\nArchivos LaTeX principales:")
print(C3_TABLES_DIR / "C3_table_main_test_comparison.tex")
print(C3_TABLES_DIR / "C3_table_selected_expost_wrc.tex")
print(C3_TABLES_DIR / "C3_table_selection_process.tex")


In [ ]:
# ============================================================
# Celda C3.8 - Comprimir y descargar resultados del Módulo C3


In [ ]:
# ============================================================

import shutil
from pathlib import Path

# Nombre del zip
zip_base_name = "C3_genetic_algorithm_results"
zip_path = Path(str(C3_OUTPUT_DIR.parent / zip_base_name) + ".zip")

# Si ya existe, lo borramos
if zip_path.exists():
    zip_path.unlink()

# Comprimir toda la carpeta de resultados C3
shutil.make_archive(
    base_name=str(C3_OUTPUT_DIR.parent / zip_base_name),
    format="zip",
    root_dir=str(C3_OUTPUT_DIR)
)

print("ZIP creado en:")
print(zip_path)

# Descargar en Colab
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("No se pudo iniciar la descarga automática.")
    print("Puedes descargar manualmente desde esta ruta:")
    print(zip_path)
    print("Error:", e)
